In [ ]:
import copy
import os
import random

import numpy as np
import torch
import torch.nn as nn
from torch.utils.data import DataLoader, Subset
from torchvision import datasets, transforms

try:
    from sklearn.metrics import balanced_accuracy_score, f1_score
except ImportError:
    balanced_accuracy_score = None
    f1_score = None


# Reproducibility
SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
torch.cuda.manual_seed_all(SEED)

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Device:", DEVICE)


# Configuration
DATA_ROOT = "/home/Paper3/E3/data"

NUM_CLASSES = 10
NUM_CLIENTS = 100

BATCH_SIZE = 64
LOCAL_EPOCHS = 1
ROUNDS = 500
LR = 0.01

NONIID_ALPHA = 0.3
TRAIN_VAL_SPLIT = 0.90

REMOVE_PERCENTAGES = [5, 10, 15, 20, 25, 30, 40, 50]
MIN_LAYER_ACTIVE_RATIO = 0.50
MIN_LAYER_ACTIVE_STRUCTURES = 1

PROTECT_FINAL_CLASSIFIER = True
PROTECT_FIRST_CONV = True
FIRST_CONV_MIN_ACTIVE_RATIO = 0.70

NUM_NEW_CLIENTS = 20
NEW_CLIENT_NONIID_ALPHA = 0.05
NEW_CLIENT_ADAPT_SAMPLES_PER_CLIENT = 500
NEW_CLIENT_EVAL_SAMPLES_PER_CLIENT = 500


# Data transforms
train_transform = transforms.Compose(
    [
        transforms.RandomCrop(32, padding=4),
        transforms.RandomHorizontalFlip(),
        transforms.ToTensor(),
        transforms.Normalize(
            mean=(0.4914, 0.4822, 0.4465),
            std=(0.2470, 0.2435, 0.2616),
        ),
    ]
)

test_transform = transforms.Compose(
    [
        transforms.ToTensor(),
        transforms.Normalize(
            mean=(0.4914, 0.4822, 0.4465),
            std=(0.2470, 0.2435, 0.2616),
        ),
    ]
)


# Load CIFAR-10 from local storage
cifar_path = os.path.join(DATA_ROOT, "cifar-10-batches-py")

print("Current directory:", os.getcwd())
print("Data root exists:", os.path.exists(DATA_ROOT))
print("CIFAR-10 data exists:", os.path.exists(cifar_path))

if not os.path.exists(cifar_path):
    raise FileNotFoundError(
        f"CIFAR-10 data was not found at: {cifar_path}\n"
        "Check the DATA_ROOT setting."
    )

train_dataset = datasets.CIFAR10(
    root=DATA_ROOT,
    train=True,
    download=False,
    transform=train_transform,
)

test_dataset = datasets.CIFAR10(
    root=DATA_ROOT,
    train=False,
    download=False,
    transform=test_transform,
)


def get_targets(dataset):
    return np.asarray(dataset.targets)


def dirichlet_partition_indices(
    targets,
    num_clients,
    alpha,
    seed=SEED,
    min_size=10,
):
    """Split sample indices across clients using a Dirichlet distribution."""
    rng = np.random.default_rng(seed)
    targets = np.asarray(targets)
    num_classes = int(targets.max()) + 1

    while True:
        client_indices = [[] for _ in range(num_clients)]

        for class_id in range(num_classes):
            class_indices = np.where(targets == class_id)[0]
            rng.shuffle(class_indices)

            proportions = rng.dirichlet(alpha * np.ones(num_clients))
            proportions /= proportions.sum()

            split_points = (
                np.cumsum(proportions) * len(class_indices)
            ).astype(int)[:-1]
            class_splits = np.split(class_indices, split_points)

            for client_id, split in enumerate(class_splits):
                client_indices[client_id].extend(split.tolist())

        if min(len(indices) for indices in client_indices) >= min_size:
            break

    for indices in client_indices:
        rng.shuffle(indices)

    return client_indices


def split_train_val_indices(
    indices,
    train_ratio=TRAIN_VAL_SPLIT,
    seed=SEED,
):
    rng = np.random.default_rng(seed)
    indices = list(indices)
    rng.shuffle(indices)

    split_index = int(train_ratio * len(indices))
    return indices[:split_index], indices[split_index:]


def make_loader(
    dataset,
    indices,
    batch_size=BATCH_SIZE,
    shuffle=True,
):
    return DataLoader(
        Subset(dataset, indices),
        batch_size=batch_size,
        shuffle=shuffle,
        num_workers=2,
        pin_memory=torch.cuda.is_available(),
    )


# Create client datasets
train_targets = get_targets(train_dataset)

client_all_indices = dirichlet_partition_indices(
    targets=train_targets,
    num_clients=NUM_CLIENTS,
    alpha=NONIID_ALPHA,
    seed=SEED,
)

client_train_indices = []
client_val_indices = []

for client_id, indices in enumerate(client_all_indices):
    train_indices, val_indices = split_train_val_indices(
        indices,
        TRAIN_VAL_SPLIT,
        seed=SEED + client_id,
    )
    client_train_indices.append(train_indices)
    client_val_indices.append(val_indices)

client_train_loaders = [
    make_loader(train_dataset, indices, shuffle=True)
    for indices in client_train_indices
]

client_val_loaders = [
    make_loader(train_dataset, indices, shuffle=False)
    for indices in client_val_indices
]

test_loader = DataLoader(
    test_dataset,
    batch_size=BATCH_SIZE,
    shuffle=False,
    num_workers=2,
    pin_memory=torch.cuda.is_available(),
)

print("CIFAR-10 training samples:", len(train_dataset))
print("CIFAR-10 test samples:", len(test_dataset))


# Model
class CIFAR10VGGSmall(nn.Module):
    """Small VGG-style network for CIFAR-10."""

    def __init__(self, num_classes=NUM_CLASSES):
        super().__init__()

        self.features = nn.Sequential(
            nn.Conv2d(3, 64, kernel_size=3, padding=1, bias=False),
            nn.BatchNorm2d(64),
            nn.ReLU(inplace=True),
            nn.Conv2d(64, 64, kernel_size=3, padding=1, bias=False),
            nn.BatchNorm2d(64),
            nn.ReLU(inplace=True),
            nn.MaxPool2d(2, 2),  # 16 x 16
            nn.Dropout2d(0.10),
            nn.Conv2d(64, 128, kernel_size=3, padding=1, bias=False),
            nn.BatchNorm2d(128),
            nn.ReLU(inplace=True),
            nn.Conv2d(128, 128, kernel_size=3, padding=1, bias=False),
            nn.BatchNorm2d(128),
            nn.ReLU(inplace=True),
            nn.MaxPool2d(2, 2),  # 8 x 8
            nn.Dropout2d(0.15),
            nn.Conv2d(128, 256, kernel_size=3, padding=1, bias=False),
            nn.BatchNorm2d(256),
            nn.ReLU(inplace=True),
            nn.Conv2d(256, 256, kernel_size=3, padding=1, bias=False),
            nn.BatchNorm2d(256),
            nn.ReLU(inplace=True),
            nn.MaxPool2d(2, 2),  # 4 x 4
            nn.Dropout2d(0.20),
        )

        self.classifier = nn.Sequential(
            nn.Flatten(),
            nn.Linear(256 * 4 * 4, 512, bias=False),
            nn.LayerNorm(512),
            nn.ReLU(inplace=True),
            nn.Dropout(0.50),
            nn.Linear(512, num_classes),
        )

    def forward(self, inputs):
        return self.classifier(self.features(inputs))


def new_model():
    return CIFAR10VGGSmall().to(DEVICE)


def clone_model(model):
    cloned_model = copy.deepcopy(model)
    return cloned_model.to(DEVICE)


def count_parameters(model):
    return sum(parameter.numel() for parameter in model.parameters())


print("VGG parameters:", count_parameters(new_model()))


# Training and evaluation
def evaluate_loss_and_metrics(model, loader):
    model.eval()
    criterion = nn.CrossEntropyLoss(reduction="sum")

    total_loss = 0.0
    total_correct = 0
    total_samples = 0
    predictions = []
    targets = []

    with torch.no_grad():
        for inputs, labels in loader:
            inputs = inputs.to(DEVICE)
            labels = labels.to(DEVICE)

            logits = model(inputs)
            loss = criterion(logits, labels)
            predicted_labels = logits.argmax(dim=1)

            total_loss += loss.item()
            total_correct += (predicted_labels == labels).sum().item()
            total_samples += labels.size(0)

            predictions.extend(predicted_labels.cpu().tolist())
            targets.extend(labels.cpu().tolist())

    average_loss = total_loss / max(total_samples, 1)
    accuracy = total_correct / max(total_samples, 1)

    if f1_score is None:
        macro_f1 = accuracy
        balanced_accuracy = accuracy
    else:
        macro_f1 = f1_score(
            targets,
            predictions,
            average="macro",
            zero_division=0,
        )
        balanced_accuracy = balanced_accuracy_score(targets, predictions)

    return {
        "loss": average_loss,
        "acc": accuracy,
        "macro_f1": macro_f1,
        "balanced_acc": balanced_accuracy,
    }


def evaluate_clients_average(model, loaders):
    metrics = [
        evaluate_loss_and_metrics(model, loader)
        for loader in loaders
    ]

    return {
        name: float(np.mean([result[name] for result in metrics]))
        for name in metrics[0]
    }


def train_supervised(
    model,
    loader,
    lr=LR,
    epochs=LOCAL_EPOCHS,
    proximal_model=None,
    mu=0.0,
):
    model.train()

    optimizer = torch.optim.SGD(
        model.parameters(),
        lr=lr,
        momentum=0.9,
        weight_decay=5e-4,
    )
    criterion = nn.CrossEntropyLoss()

    for _ in range(epochs):
        for inputs, labels in loader:
            inputs = inputs.to(DEVICE)
            labels = labels.to(DEVICE)

            optimizer.zero_grad(set_to_none=True)

            logits = model(inputs)
            loss = criterion(logits, labels)

            if proximal_model is not None and mu > 0:
                proximal_penalty = torch.tensor(0.0, device=DEVICE)

                for parameter, reference in zip(
                    model.parameters(),
                    proximal_model.parameters(),
                ):
                    proximal_penalty += torch.sum(
                        (parameter - reference.detach()) ** 2
                    )

                loss += (mu / 2.0) * proximal_penalty

            loss.backward()
            optimizer.step()

    model.eval()
    return model


def average_models(models, weights=None):
    if not models:
        raise ValueError("At least one model is required.")

    if weights is None:
        weights = np.ones(len(models), dtype=np.float64)
    else:
        weights = np.asarray(weights, dtype=np.float64)

    if len(weights) != len(models):
        raise ValueError("The number of weights must match the models.")

    if weights.sum() <= 0:
        raise ValueError("Model weights must have a positive sum.")

    weights /= weights.sum()

    averaged_model = clone_model(models[0])
    averaged_state = copy.deepcopy(averaged_model.state_dict())

    for name, value in averaged_state.items():
        if torch.is_floating_point(value):
            averaged_state[name] = torch.zeros_like(value)

    for model, weight in zip(models, weights):
        state = model.state_dict()

        for name, value in averaged_state.items():
            if torch.is_floating_point(value):
                averaged_state[name] += (
                    state[name].to(value.device) * float(weight)
                )
            else:
                averaged_state[name] = state[name].clone()

    averaged_model.load_state_dict(averaged_state)
    averaged_model.to(DEVICE)
    averaged_model.eval()

    return averaged_model


# Checkpoints
def save_checkpoint(
    path,
    model,
    history,
    final_test_metrics,
    config,
):
    torch.save(
        {
            "model_state_dict": model.state_dict(),
            "history": history,
            "final_test_metrics": final_test_metrics,
            "config": config,
        },
        path,
    )


def load_full_model_checkpoint(path):
    checkpoint = torch.load(path, map_location=DEVICE)

    model = new_model()
    model.load_state_dict(checkpoint["model_state_dict"])
    model.to(DEVICE)
    model.eval()

    return model, checkpoint


In [ ]:
# ============================================================
# CELL 2: FedAvg Baseline Training
# ============================================================

FEDAVG_PATH = "FedAvg_CIFAR10_VGG.pt"


def run_fedavg(
    rounds=ROUNDS,
    local_epochs=LOCAL_EPOCHS,
    lr=LR,
    client_fraction=1.0,
    early_stop_patience=5,
    min_delta=1e-4,
    save_path=FEDAVG_PATH,
):
    global_model = new_model()

    history = {
        "round": [],
        "train_loss": [],
        "train_acc": [],
        "val_loss": [],
        "val_acc": [],
        "best_round": None,
    }

    best_val_acc = -float("inf")
    best_state = copy.deepcopy(global_model.state_dict())
    best_round = 0
    rounds_without_improvement = 0

    for round_number in range(1, rounds + 1):
        num_selected_clients = max(
            1,
            int(client_fraction * NUM_CLIENTS),
        )

        selected_clients = np.random.choice(
            NUM_CLIENTS,
            size=num_selected_clients,
            replace=False,
        )

        local_models = []
        local_weights = []

        # Train the selected clients
        for client_id in selected_clients:
            local_model = clone_model(global_model)

            local_model = train_supervised(
                model=local_model,
                loader=client_train_loaders[client_id],
                lr=lr,
                epochs=local_epochs,
            )

            local_models.append(local_model)
            local_weights.append(
                len(client_train_indices[client_id])
            )

        # Aggregate the local models
        global_model = average_models(
            local_models,
            local_weights,
        )

        train_metrics = evaluate_clients_average(
            global_model,
            client_train_loaders,
        )
        val_metrics = evaluate_clients_average(
            global_model,
            client_val_loaders,
        )

        print(
            f"FedAvg | Round {round_number:03d} | "
            f"Train Acc: {train_metrics['acc']:.4f} | "
            f"Val Acc: {val_metrics['acc']:.4f}"
        )

        history["round"].append(round_number)
        history["train_loss"].append(train_metrics["loss"])
        history["train_acc"].append(train_metrics["acc"])
        history["val_loss"].append(val_metrics["loss"])
        history["val_acc"].append(val_metrics["acc"])

        # Keep the best validation model
        if val_metrics["acc"] > best_val_acc + min_delta:
            best_val_acc = val_metrics["acc"]
            best_state = copy.deepcopy(
                global_model.state_dict()
            )
            best_round = round_number
            rounds_without_improvement = 0
        else:
            rounds_without_improvement += 1

        if rounds_without_improvement >= early_stop_patience:
            print(
                f"Early stopping at round {round_number}. "
                f"Best round: {best_round}"
            )
            break

    # Restore the best model
    global_model.load_state_dict(best_state)
    global_model.to(DEVICE)
    global_model.eval()

    final_test_metrics = evaluate_loss_and_metrics(
        global_model,
        test_loader,
    )

    history["best_round"] = best_round
    history["test_loss"] = final_test_metrics["loss"]
    history["test_acc"] = final_test_metrics["acc"]
    history["test_macro_f1"] = final_test_metrics["macro_f1"]
    history["test_balanced_acc"] = final_test_metrics[
        "balanced_acc"
    ]

    print("FedAvg final test:", final_test_metrics)

    save_checkpoint(
        path=save_path,
        model=global_model,
        history=history,
        final_test_metrics=final_test_metrics,
        config={
            "method": "FedAvg",
            "dataset": "CIFAR10",
            "architecture": "VGG-style",
            "rounds": rounds,
            "local_epochs": local_epochs,
            "lr": lr,
            "client_fraction": client_fraction,
        },
    )

    print("Saved:", save_path)

    return global_model, history, final_test_metrics


fedavg_model, fedavg_history, fedavg_test_metrics = run_fedavg()

In [ ]:
# ============================================================
# CELL 3: FedProx Baseline Training
# ============================================================

FEDPROX_PATH = "FedProx_CIFAR10_VGG.pt"
FEDPROX_MU = 0.01


def run_fedprox(
    rounds=ROUNDS,
    local_epochs=LOCAL_EPOCHS,
    lr=LR,
    mu=FEDPROX_MU,
    client_fraction=1.0,
    early_stop_patience=5,
    min_delta=1e-4,
    save_path=FEDPROX_PATH,
):
    global_model = new_model()

    history = {
        "round": [],
        "train_loss": [],
        "train_acc": [],
        "val_loss": [],
        "val_acc": [],
        "best_round": None,
    }

    best_val_acc = -float("inf")
    best_state = copy.deepcopy(global_model.state_dict())
    best_round = 0
    rounds_without_improvement = 0

    for round_number in range(1, rounds + 1):
        # Pick the clients used in this round
        num_selected_clients = max(
            1,
            int(client_fraction * NUM_CLIENTS),
        )

        selected_clients = np.random.choice(
            NUM_CLIENTS,
            size=num_selected_clients,
            replace=False,
        )

        local_models = []
        local_weights = []

        for client_id in selected_clients:
            local_model = clone_model(global_model)
            reference_model = clone_model(global_model)

            local_model = train_supervised(
                model=local_model,
                loader=client_train_loaders[client_id],
                lr=lr,
                epochs=local_epochs,
                proximal_model=reference_model,
                mu=mu,
            )

            local_models.append(local_model)
            local_weights.append(
                len(client_train_indices[client_id])
            )

        # Combine the client models
        global_model = average_models(
            local_models,
            local_weights,
        )

        train_metrics = evaluate_clients_average(
            global_model,
            client_train_loaders,
        )

        val_metrics = evaluate_clients_average(
            global_model,
            client_val_loaders,
        )

        print(
            f"FedProx | Round {round_number:03d} | "
            f"Train Acc: {train_metrics['acc']:.4f} | "
            f"Val Acc: {val_metrics['acc']:.4f}"
        )

        history["round"].append(round_number)
        history["train_loss"].append(train_metrics["loss"])
        history["train_acc"].append(train_metrics["acc"])
        history["val_loss"].append(val_metrics["loss"])
        history["val_acc"].append(val_metrics["acc"])

        # Save the best validation result
        if val_metrics["acc"] > best_val_acc + min_delta:
            best_val_acc = val_metrics["acc"]
            best_state = copy.deepcopy(
                global_model.state_dict()
            )
            best_round = round_number
            rounds_without_improvement = 0
        else:
            rounds_without_improvement += 1

        if rounds_without_improvement >= early_stop_patience:
            print(
                f"Early stopping at round {round_number}. "
                f"Best round: {best_round}"
            )
            break

    # Use the best model found during training
    global_model.load_state_dict(best_state)
    global_model.to(DEVICE)
    global_model.eval()

    final_test_metrics = evaluate_loss_and_metrics(
        global_model,
        test_loader,
    )

    history["best_round"] = best_round
    history["test_loss"] = final_test_metrics["loss"]
    history["test_acc"] = final_test_metrics["acc"]
    history["test_macro_f1"] = final_test_metrics["macro_f1"]
    history["test_balanced_acc"] = final_test_metrics[
        "balanced_acc"
    ]

    print("FedProx final test:", final_test_metrics)

    save_checkpoint(
        path=save_path,
        model=global_model,
        history=history,
        final_test_metrics=final_test_metrics,
        config={
            "method": "FedProx",
            "dataset": "CIFAR10",
            "architecture": "VGG-style",
            "mu": mu,
            "rounds": rounds,
            "local_epochs": local_epochs,
            "lr": lr,
            "client_fraction": client_fraction,
        },
    )

    print("Saved:", save_path)

    return global_model, history, final_test_metrics


fedprox_model, fedprox_history, fedprox_test_metrics = run_fedprox()

In [ ]:
# ============================================================
# CELL 4: RAS-FL Training
# Runtime-Aware Shrinkable Federated Learning
# ============================================================

import math

import torch.nn.functional as F


RAS_FL_PATH = "RAS_FL_CIFAR10_VGG.pt"

IMPORTANCE_BETA = 0.95
IMPORTANCE_MAX_BATCHES = 5
IMPORTANCE_STABILIZATION_ROUNDS = 10
IMPORTANCE_GAMMA = 0.0003

SHRINKAGE_AWARE_TRAINING = True
SHRINKAGE_WARMUP_ROUNDS = 15
MASK_TRAINING_PROB = 0.25

USE_SELF_DISTILLATION = True
DISTILLATION_ALPHA = 0.20
DISTILLATION_TEMPERATURE = 2.0

EARLY_STOP_START_ROUND = 35
VALIDATION_MASK_LEVELS = [10, 20, 30, 40]
MASKED_VAL_EVAL_EVERY = 1
FULL_VAL_WEIGHT = 0.50
MASKED_VAL_WEIGHT = 0.50

RAS_FL_EARLY_STOP_PATIENCE = 5
RAS_FL_MIN_DELTA = 1e-4


def get_structurable_layers(model):
    return [
        (name, module)
        for name, module in model.named_modules()
        if isinstance(module, (nn.Conv2d, nn.Linear))
    ]


def get_final_structurable_layer_name(model):
    layers = get_structurable_layers(model)
    return layers[-1][0] if layers else None


def get_first_conv_layer_name(model):
    for name, module in model.named_modules():
        if isinstance(module, nn.Conv2d):
            return name

    return None


def initialize_structured_importance(model, value=0.0):
    importance = {}

    for name, module in get_structurable_layers(model):
        if isinstance(module, nn.Conv2d):
            size = module.out_channels
        else:
            size = module.out_features

        importance[name] = torch.full(
            (size,),
            value,
            device=DEVICE,
        )

    return importance


def estimate_local_structured_importance(
    model,
    loader,
    max_batches=IMPORTANCE_MAX_BATCHES,
):
    model.train()

    criterion = nn.CrossEntropyLoss()
    importance = initialize_structured_importance(model)
    modules = dict(model.named_modules())
    batches_used = 0

    for inputs, labels in loader:
        inputs = inputs.to(DEVICE)
        labels = labels.to(DEVICE)

        model.zero_grad(set_to_none=True)

        logits = model(inputs)
        loss = criterion(logits, labels)
        loss.backward()

        with torch.no_grad():
            for name in importance:
                module = modules[name]

                if module.weight.grad is None:
                    continue

                scores = torch.abs(
                    module.weight.data
                    * module.weight.grad.data
                )

                if isinstance(module, nn.Conv2d):
                    scores = scores.sum(dim=(1, 2, 3))
                else:
                    scores = scores.sum(dim=1)

                importance[name] += scores

        batches_used += 1

        if batches_used >= max_batches:
            break

    if batches_used:
        for name in importance:
            importance[name] /= batches_used

    model.eval()
    return importance


def update_global_structured_importance_ema(
    global_importance,
    client_importances,
    client_weights,
):
    if not client_importances:
        return global_importance

    total_weight = float(np.sum(client_weights))

    for name in global_importance:
        weighted_average = torch.zeros_like(
            global_importance[name]
        )

        for importance, weight in zip(
            client_importances,
            client_weights,
        ):
            weighted_average += (
                weight / total_weight
            ) * importance[name]

        global_importance[name] = (
            IMPORTANCE_BETA * global_importance[name]
            + (1.0 - IMPORTANCE_BETA)
            * weighted_average
        )

    return global_importance


def get_layer_min_count(
    model,
    layer_name,
    module,
    size,
):
    final_layer = get_final_structurable_layer_name(model)
    first_conv = get_first_conv_layer_name(model)

    if (
        PROTECT_FINAL_CLASSIFIER
        and layer_name == final_layer
    ):
        active_ratio = 1.0
    elif (
        PROTECT_FIRST_CONV
        and layer_name == first_conv
    ):
        active_ratio = max(
            MIN_LAYER_ACTIVE_RATIO,
            FIRST_CONV_MIN_ACTIVE_RATIO,
        )
    else:
        active_ratio = MIN_LAYER_ACTIVE_RATIO

    minimum = max(
        MIN_LAYER_ACTIVE_STRUCTURES,
        int(math.ceil(active_ratio * size)),
    )

    return min(size, minimum)


def build_full_structured_mask(model):
    structured_mask = {}

    for name, module in get_structurable_layers(model):
        if isinstance(module, nn.Conv2d):
            size = module.out_channels
        else:
            size = module.out_features

        structured_mask[name] = torch.ones(
            size,
            device=DEVICE,
        )

    return structured_mask


def build_boundary_aware_remove_low_mask(
    model,
    importance,
    remove_percentage,
):
    structured_mask = build_full_structured_mask(model)
    final_layer = get_final_structurable_layer_name(model)

    candidates = []
    active_counts = {}
    minimum_counts = {}
    total_prunable = 0

    for name, module in get_structurable_layers(model):
        scores = (
            importance[name]
            .detach()
            .flatten()
            .to(DEVICE)
        )

        size = scores.numel()

        active_counts[name] = size
        minimum_counts[name] = get_layer_min_count(
            model,
            name,
            module,
            size,
        )

        if (
            PROTECT_FINAL_CLASSIFIER
            and name == final_layer
        ):
            continue

        total_prunable += size

        for index in range(size):
            candidates.append(
                (
                    float(scores[index].item()),
                    name,
                    index,
                )
            )

    candidates.sort(key=lambda item: item[0])

    target_remove = int(
        round(
            (remove_percentage / 100.0)
            * total_prunable
        )
    )

    removed = 0

    for _, name, index in candidates:
        if removed >= target_remove:
            break

        if active_counts[name] <= minimum_counts[name]:
            continue

        structured_mask[name][index] = 0.0
        active_counts[name] -= 1
        removed += 1

    metadata = {
        "requested_remove_percentage": remove_percentage,
        "actual_removed": removed,
        "actual_removed_ratio": (
            removed / max(total_prunable, 1)
        ),
        "total_prunable_structures": total_prunable,
        "uses_importance": True,
        "uses_personalization": False,
        "min_layer_active_ratio": (
            MIN_LAYER_ACTIVE_RATIO
        ),
        "global_budget_based_removal": True,
    }

    return structured_mask, metadata


def build_boundary_aware_importance_roadmap(
    model,
    importance,
    remove_percentages=REMOVE_PERCENTAGES,
):
    roadmap = {
        "full": build_full_structured_mask(model)
    }

    metadata = {
        "full": {
            "requested_remove_percentage": 0,
            "actual_removed": 0,
            "actual_removed_ratio": 0.0,
            "uses_importance": True,
        }
    }

    for percentage in remove_percentages:
        mode = f"remove_low_{percentage}"

        roadmap[mode], metadata[mode] = (
            build_boundary_aware_remove_low_mask(
                model,
                importance,
                percentage,
            )
        )

    return roadmap, metadata


def expand_structured_masks_to_parameter_masks(
    model,
    structured_masks,
):
    parameter_masks = {}
    previous_feature_mask = None

    for layer_name, module in model.named_modules():
        if (
            isinstance(module, nn.Conv2d)
            and layer_name in structured_masks
        ):
            mask = structured_masks[layer_name].to(
                DEVICE
            )

            parameter_masks[
                f"{layer_name}.weight"
            ] = mask.view(
                -1,
                1,
                1,
                1,
            ).expand_as(module.weight.data)

            if module.bias is not None:
                parameter_masks[
                    f"{layer_name}.bias"
                ] = mask.view(-1).expand_as(
                    module.bias.data
                )

            previous_feature_mask = mask

        elif isinstance(module, nn.BatchNorm2d):
            if (
                previous_feature_mask is not None
                and module.num_features
                == previous_feature_mask.numel()
            ):
                mask = previous_feature_mask.to(
                    DEVICE
                )

                if module.weight is not None:
                    parameter_masks[
                        f"{layer_name}.weight"
                    ] = mask.view(-1).expand_as(
                        module.weight.data
                    )

                if module.bias is not None:
                    parameter_masks[
                        f"{layer_name}.bias"
                    ] = mask.view(-1).expand_as(
                        module.bias.data
                    )

        elif (
            isinstance(module, nn.Linear)
            and layer_name in structured_masks
        ):
            mask = structured_masks[layer_name].to(
                DEVICE
            )

            parameter_masks[
                f"{layer_name}.weight"
            ] = mask.view(
                -1,
                1,
            ).expand_as(module.weight.data)

            if module.bias is not None:
                parameter_masks[
                    f"{layer_name}.bias"
                ] = mask.view(-1).expand_as(
                    module.bias.data
                )

            previous_feature_mask = mask

        elif isinstance(module, nn.LayerNorm):
            if (
                previous_feature_mask is not None
                and len(module.normalized_shape) == 1
                and module.normalized_shape[0]
                == previous_feature_mask.numel()
            ):
                mask = previous_feature_mask.to(
                    DEVICE
                )

                if module.elementwise_affine:
                    parameter_masks[
                        f"{layer_name}.weight"
                    ] = mask.view(-1).expand_as(
                        module.weight.data
                    )

                    parameter_masks[
                        f"{layer_name}.bias"
                    ] = mask.view(-1).expand_as(
                        module.bias.data
                    )

    return parameter_masks


def apply_parameter_masks_to_model(
    model,
    parameter_masks,
):
    with torch.no_grad():
        for name, parameter in model.named_parameters():
            if name in parameter_masks:
                parameter.data.mul_(
                    parameter_masks[name].to(
                        parameter.device
                    )
                )


def mask_gradients(model, parameter_masks):
    with torch.no_grad():
        for name, parameter in model.named_parameters():
            if (
                name in parameter_masks
                and parameter.grad is not None
            ):
                parameter.grad.mul_(
                    parameter_masks[name].to(
                        parameter.device
                    )
                )


def compute_distillation_loss(
    student_logits,
    teacher_logits,
    temperature,
):
    return F.kl_div(
        F.log_softmax(
            student_logits / temperature,
            dim=1,
        ),
        F.softmax(
            teacher_logits / temperature,
            dim=1,
        ),
        reduction="batchmean",
    ) * (temperature ** 2)


def get_training_mask_levels_by_round(
    round_number,
):
    if round_number <= SHRINKAGE_WARMUP_ROUNDS:
        return []

    if round_number <= 15:
        return [5, 10]

    if round_number <= 30:
        return [5, 10, 15, 20]

    if round_number <= 50:
        return [5, 10, 15, 20, 25, 30]

    return [5, 10, 15, 20, 25, 30, 40]


def evaluate_masked_clients_average(
    model,
    loaders,
    global_importance,
    remove_percentages=VALIDATION_MASK_LEVELS,
):
    accuracies = []

    for percentage in remove_percentages:
        structured_mask, _ = (
            build_boundary_aware_remove_low_mask(
                model,
                global_importance,
                percentage,
            )
        )

        masked_model = clone_model(model)

        parameter_masks = (
            expand_structured_masks_to_parameter_masks(
                masked_model,
                structured_mask,
            )
        )

        apply_parameter_masks_to_model(
            masked_model,
            parameter_masks,
        )

        masked_model.eval()

        metrics = evaluate_clients_average(
            masked_model,
            loaders,
        )

        accuracies.append(metrics["acc"])

    return float(np.mean(accuracies))


def train_one_client_ras_fl(
    local_model,
    loader,
    lr,
    epochs,
    round_number,
    global_importance,
):
    local_model.train()

    teacher_model = clone_model(local_model)
    teacher_model.eval()

    for parameter in teacher_model.parameters():
        parameter.requires_grad = False

    optimizer = torch.optim.SGD(
        local_model.parameters(),
        lr=lr,
        momentum=0.9,
        weight_decay=5e-4,
    )

    criterion = nn.CrossEntropyLoss()

    mask_levels = get_training_mask_levels_by_round(
        round_number
    )

    use_masks = (
        SHRINKAGE_AWARE_TRAINING
        and round_number > SHRINKAGE_WARMUP_ROUNDS
        and bool(mask_levels)
    )

    for _ in range(epochs):
        for inputs, labels in loader:
            inputs = inputs.to(DEVICE)
            labels = labels.to(DEVICE)

            use_masked_batch = (
                use_masks
                and random.random()
                < MASK_TRAINING_PROB
            )

            optimizer.zero_grad(set_to_none=True)

            if not use_masked_batch:
                logits = local_model(inputs)
                loss = criterion(logits, labels)

                loss.backward()
                optimizer.step()
                continue

            # Train a smaller version of the model
            remove_percentage = random.choice(
                mask_levels
            )

            structured_mask, _ = (
                build_boundary_aware_remove_low_mask(
                    local_model,
                    global_importance,
                    remove_percentage,
                )
            )

            parameter_masks = (
                expand_structured_masks_to_parameter_masks(
                    local_model,
                    structured_mask,
                )
            )

            parameter_backup = {
                name: parameter.data.clone()
                for name, parameter
                in local_model.named_parameters()
                if name in parameter_masks
            }

            apply_parameter_masks_to_model(
                local_model,
                parameter_masks,
            )

            student_logits = local_model(inputs)

            classification_loss = criterion(
                student_logits,
                labels,
            )

            if USE_SELF_DISTILLATION:
                with torch.no_grad():
                    teacher_logits = teacher_model(
                        inputs
                    )

                distillation_loss = (
                    compute_distillation_loss(
                        student_logits,
                        teacher_logits,
                        DISTILLATION_TEMPERATURE,
                    )
                )

                loss = (
                    (1.0 - DISTILLATION_ALPHA)
                    * classification_loss
                    + DISTILLATION_ALPHA
                    * distillation_loss
                )
            else:
                loss = classification_loss

            loss.backward()

            # Restore the full model before updating it
            with torch.no_grad():
                for name, parameter in (
                    local_model.named_parameters()
                ):
                    if name in parameter_backup:
                        parameter.data.copy_(
                            parameter_backup[name]
                        )

            mask_gradients(
                local_model,
                parameter_masks,
            )

            optimizer.step()

    local_model.eval()
    return local_model


def convert_roadmap_to_cpu(roadmap):
    return {
        mode: {
            name: mask.detach().cpu()
            for name, mask in layer_masks.items()
        }
        for mode, layer_masks in roadmap.items()
    }


def move_roadmap_to_device(
    roadmap,
    device=DEVICE,
):
    return {
        mode: {
            name: mask.to(device)
            for name, mask in layer_masks.items()
        }
        for mode, layer_masks in roadmap.items()
    }


def count_active_structures(structured_mask):
    active = 0
    total = 0

    for mask in structured_mask.values():
        active += int(mask.sum().item())
        total += int(mask.numel())

    active_ratio = active / max(total, 1)

    return active, total, active_ratio


def run_ras_fl(
    rounds=ROUNDS,
    local_epochs=LOCAL_EPOCHS,
    lr=LR,
    client_fraction=1.0,
    early_stop_patience=RAS_FL_EARLY_STOP_PATIENCE,
    min_delta=RAS_FL_MIN_DELTA,
    save_path=RAS_FL_PATH,
):
    global_model = new_model()

    global_importance = (
        initialize_structured_importance(
            global_model,
            value=1e-6,
        )
    )

    history = {
        "round": [],
        "train_loss": [],
        "train_acc": [],
        "val_loss": [],
        "val_acc": [],
        "masked_val_acc": [],
        "runtime_selection_score": [],
        "best_round": None,
    }

    best_score = -float("inf")
    best_state = copy.deepcopy(
        global_model.state_dict()
    )
    best_importance = copy.deepcopy(
        global_importance
    )
    best_round = 0
    rounds_without_improvement = 0

    for round_number in range(1, rounds + 1):
        num_selected_clients = max(
            1,
            int(client_fraction * NUM_CLIENTS),
        )

        selected_clients = np.random.choice(
            NUM_CLIENTS,
            size=num_selected_clients,
            replace=False,
        )

        local_models = []
        local_weights = []
        local_importances = []
        importance_weights = []

        # Estimate importance and train each client
        for client_id in selected_clients:
            local_model = clone_model(global_model)

            local_importance = (
                estimate_local_structured_importance(
                    clone_model(global_model),
                    client_train_loaders[client_id],
                    max_batches=(
                        IMPORTANCE_MAX_BATCHES
                    ),
                )
            )

            local_model = train_one_client_ras_fl(
                local_model=local_model,
                loader=(
                    client_train_loaders[client_id]
                ),
                lr=lr,
                epochs=local_epochs,
                round_number=round_number,
                global_importance=global_importance,
            )

            client_size = len(
                client_train_indices[client_id]
            )

            local_models.append(local_model)
            local_weights.append(client_size)
            local_importances.append(
                local_importance
            )
            importance_weights.append(client_size)

        global_model = average_models(
            local_models,
            local_weights,
        )

        global_importance = (
            update_global_structured_importance_ema(
                global_importance,
                local_importances,
                importance_weights,
            )
        )

        train_metrics = evaluate_clients_average(
            global_model,
            client_train_loaders,
        )

        val_metrics = evaluate_clients_average(
            global_model,
            client_val_loaders,
        )

        full_val_acc = val_metrics["acc"]

        if (
            round_number > SHRINKAGE_WARMUP_ROUNDS
            and round_number
            % MASKED_VAL_EVAL_EVERY == 0
        ):
            masked_val_acc = (
                evaluate_masked_clients_average(
                    global_model,
                    client_val_loaders,
                    global_importance,
                    remove_percentages=(
                        VALIDATION_MASK_LEVELS
                    ),
                )
            )
        else:
            masked_val_acc = full_val_acc

        runtime_score = (
            FULL_VAL_WEIGHT * full_val_acc
            + MASKED_VAL_WEIGHT
            * masked_val_acc
        )

        history["round"].append(round_number)
        history["train_loss"].append(
            train_metrics["loss"]
        )
        history["train_acc"].append(
            train_metrics["acc"]
        )
        history["val_loss"].append(
            val_metrics["loss"]
        )
        history["val_acc"].append(full_val_acc)
        history["masked_val_acc"].append(
            masked_val_acc
        )
        history["runtime_selection_score"].append(
            runtime_score
        )

        print(
            f"RAS-FL | Round {round_number:03d} | "
            f"Masks: "
            f"{get_training_mask_levels_by_round(round_number)} | "
            f"Train Acc: {train_metrics['acc']:.4f} | "
            f"Full Val Acc: {full_val_acc:.4f} | "
            f"Masked Val Acc: {masked_val_acc:.4f} | "
            f"Runtime Score: {runtime_score:.4f}"
        )

        # Keep the model that works best in both modes
        if runtime_score > best_score + min_delta:
            best_score = runtime_score

            best_state = copy.deepcopy(
                global_model.state_dict()
            )

            best_importance = copy.deepcopy(
                global_importance
            )

            best_round = round_number
            rounds_without_improvement = 0

            print(
                f"RAS-FL | New best runtime score: "
                f"{best_score:.4f} "
                f"at round {best_round}"
            )

        elif round_number >= EARLY_STOP_START_ROUND:
            rounds_without_improvement += 1

        else:
            rounds_without_improvement = 0

        if (
            round_number >= EARLY_STOP_START_ROUND
            and rounds_without_improvement
            >= early_stop_patience
        ):
            print(
                f"RAS-FL early stopping at round "
                f"{round_number}. "
                f"Best round: {best_round}"
            )
            break

    # Return to the best training round
    global_model.load_state_dict(best_state)
    global_model.to(DEVICE)
    global_model.eval()

    global_importance = best_importance

    history["best_round"] = best_round
    history["best_runtime_selection_score"] = (
        best_score
    )

    # Recalculate importance using all clients
    for step in range(
        IMPORTANCE_STABILIZATION_ROUNDS
    ):
        local_importances = []
        importance_weights = []

        for client_id in range(NUM_CLIENTS):
            local_importance = (
                estimate_local_structured_importance(
                    clone_model(global_model),
                    client_train_loaders[client_id],
                    max_batches=(
                        IMPORTANCE_MAX_BATCHES
                    ),
                )
            )

            local_importances.append(
                local_importance
            )

            importance_weights.append(
                len(client_train_indices[client_id])
            )

        global_importance = (
            update_global_structured_importance_ema(
                global_importance,
                local_importances,
                importance_weights,
            )
        )

        print(
            f"RAS-FL importance stabilization "
            f"{step + 1}/"
            f"{IMPORTANCE_STABILIZATION_ROUNDS}"
        )

    final_test_metrics = (
        evaluate_loss_and_metrics(
            global_model,
            test_loader,
        )
    )

    roadmap, roadmap_metadata = (
        build_boundary_aware_importance_roadmap(
            global_model,
            global_importance,
            REMOVE_PERCENTAGES,
        )
    )

    print(
        "RAS-FL final full-model test:",
        final_test_metrics,
    )

    torch.save(
        {
            "model_state_dict": (
                global_model.state_dict()
            ),
            "global_structured_importance": {
                name: tensor.detach().cpu()
                for name, tensor
                in global_importance.items()
            },
            "mask_roadmap": (
                convert_roadmap_to_cpu(roadmap)
            ),
            "mask_roadmap_metadata": (
                roadmap_metadata
            ),
            "history": history,
            "final_test_metrics": (
                final_test_metrics
            ),
            "config": {
                "method": "RAS-FL",
                "dataset": "CIFAR10",
                "architecture": "VGG-style",
                "remove_percentages": (
                    REMOVE_PERCENTAGES
                ),
                "min_layer_active_ratio": (
                    MIN_LAYER_ACTIVE_RATIO
                ),
                "uses_importance": True,
                "uses_personalization": True,
                "runtime_aware_validation": True,
            },
        },
        save_path,
    )

    print("Saved:", save_path)

    return (
        global_model,
        global_importance,
        roadmap,
        roadmap_metadata,
        history,
        final_test_metrics,
    )


(
    ras_fl_model,
    global_structured_importance,
    ras_fl_roadmap,
    ras_fl_roadmap_metadata,
    ras_fl_history,
    ras_fl_test_metrics,
) = run_ras_fl()


def load_ras_fl_checkpoint(
    path=RAS_FL_PATH,
):
    checkpoint = torch.load(
        path,
        map_location=DEVICE,
    )

    model = new_model()

    model.load_state_dict(
        checkpoint["model_state_dict"]
    )

    model.to(DEVICE)
    model.eval()

    importance = {
        name: tensor.to(DEVICE)
        for name, tensor
        in checkpoint[
            "global_structured_importance"
        ].items()
    }

    roadmap = move_roadmap_to_device(
        checkpoint["mask_roadmap"],
        DEVICE,
    )

    return model, importance, roadmap, checkpoint



In [ ]:
# ============================================================
# CELL 5: Full-Model Baseline Evaluation
# ============================================================

def evaluate_saved_full_models():
    rows = []

    model_paths = [
        ("FedAvg", FEDAVG_PATH),
        ("FedProx", FEDPROX_PATH),
        ("RAS-FL Full Model", RAS_FL_PATH),
    ]

    for method_name, model_path in model_paths:
        if not os.path.exists(model_path):
            print("Missing:", model_path)
            continue

        # RAS-FL uses a different checkpoint format
        if method_name == "RAS-FL Full Model":
            model, _, _, _ = load_ras_fl_checkpoint(
                model_path
            )
        else:
            model, _ = load_full_model_checkpoint(
                model_path
            )

        metrics = evaluate_loss_and_metrics(
            model,
            test_loader,
        )

        rows.append(
            {
                "method": method_name,
                "accuracy": metrics["acc"],
                "loss": metrics["loss"],
                "macro_f1": metrics["macro_f1"],
                "balanced_acc": metrics["balanced_acc"],
            }
        )

    results_df = pd.DataFrame(rows)

    display(results_df)

    results_df.to_csv(
        "cifar10_vgg_full_model_baseline_results.csv",
        index=False,
    )

    return results_df


full_model_baseline_df = evaluate_saved_full_models()

In [ ]:
# ============================================================
# CELL 6: FedDrop Baseline Training
# ============================================================

FEDDROP_PATH = "FedDrop_CIFAR10_VGG.pt"


def build_random_boundary_mask(
    model,
    remove_percentage,
    seed,
):
    rng = np.random.default_rng(seed)

    structured_mask = build_full_structured_mask(model)
    final_layer = get_final_structurable_layer_name(model)

    candidates = []
    active_counts = {}
    minimum_counts = {}
    total_prunable = 0

    for name, module in get_structurable_layers(model):
        if isinstance(module, nn.Conv2d):
            size = module.out_channels
        else:
            size = module.out_features

        active_counts[name] = size
        minimum_counts[name] = get_layer_min_count(
            model,
            name,
            module,
            size,
        )

        if (
            PROTECT_FINAL_CLASSIFIER
            and name == final_layer
        ):
            continue

        total_prunable += size

        for index in range(size):
            candidates.append((name, index))

    rng.shuffle(candidates)

    target_remove = int(
        round(
            (remove_percentage / 100.0)
            * total_prunable
        )
    )

    removed = 0

    # Remove random structures without crossing layer limits
    for name, index in candidates:
        if removed >= target_remove:
            break

        if active_counts[name] <= minimum_counts[name]:
            continue

        structured_mask[name][index] = 0.0
        active_counts[name] -= 1
        removed += 1

    metadata = {
        "requested_remove_percentage": remove_percentage,
        "actual_removed": removed,
        "actual_removed_ratio": (
            removed / max(total_prunable, 1)
        ),
        "uses_importance": False,
        "uses_personalization": False,
        "mask_type": "random_boundary",
    }

    return structured_mask, metadata


def build_feddrop_roadmap(model):
    roadmap = {
        "full": build_full_structured_mask(model)
    }

    metadata = {
        "full": {
            "requested_remove_percentage": 0,
            "actual_removed_ratio": 0.0,
        }
    }

    for percentage in REMOVE_PERCENTAGES:
        mode = f"feddrop_remove_{percentage}"

        roadmap[mode], metadata[mode] = (
            build_random_boundary_mask(
                model,
                percentage,
                seed=SEED + 600 + percentage,
            )
        )

    return roadmap, metadata


def train_one_client_feddrop(
    local_model,
    loader,
    lr,
    epochs,
):
    local_model.train()

    optimizer = torch.optim.SGD(
        local_model.parameters(),
        lr=lr,
        momentum=0.9,
        weight_decay=5e-4,
    )

    criterion = nn.CrossEntropyLoss()

    for _ in range(epochs):
        for inputs, labels in loader:
            inputs = inputs.to(DEVICE)
            labels = labels.to(DEVICE)

            remove_percentage = random.choice(
                REMOVE_PERCENTAGES
            )

            structured_mask, _ = (
                build_random_boundary_mask(
                    local_model,
                    remove_percentage,
                    seed=random.randint(0, 10**9),
                )
            )

            parameter_masks = (
                expand_structured_masks_to_parameter_masks(
                    local_model,
                    structured_mask,
                )
            )

            parameter_backup = {
                name: parameter.data.clone()
                for name, parameter
                in local_model.named_parameters()
                if name in parameter_masks
            }

            optimizer.zero_grad(set_to_none=True)

            apply_parameter_masks_to_model(
                local_model,
                parameter_masks,
            )

            logits = local_model(inputs)
            loss = criterion(logits, labels)

            loss.backward()

            # Restore the full model before the update
            with torch.no_grad():
                for name, parameter in (
                    local_model.named_parameters()
                ):
                    if name in parameter_backup:
                        parameter.data.copy_(
                            parameter_backup[name]
                        )

            mask_gradients(
                local_model,
                parameter_masks,
            )

            optimizer.step()

    local_model.eval()
    return local_model


def run_feddrop(
    rounds=ROUNDS,
    local_epochs=LOCAL_EPOCHS,
    lr=LR,
    early_stop_patience=5,
    min_delta=1e-4,
    save_path=FEDDROP_PATH,
):
    global_model = new_model()

    history = {
        "round": [],
        "train_acc": [],
        "val_acc": [],
        "best_round": None,
    }

    best_val_acc = -float("inf")
    best_state = copy.deepcopy(
        global_model.state_dict()
    )
    best_round = 0
    rounds_without_improvement = 0

    for round_number in range(1, rounds + 1):
        local_models = []
        local_weights = []

        # Train every client with a random submodel
        for client_id in range(NUM_CLIENTS):
            local_model = clone_model(global_model)

            local_model = train_one_client_feddrop(
                local_model=local_model,
                loader=client_train_loaders[client_id],
                lr=lr,
                epochs=local_epochs,
            )

            local_models.append(local_model)
            local_weights.append(
                len(client_train_indices[client_id])
            )

        global_model = average_models(
            local_models,
            local_weights,
        )

        train_metrics = evaluate_clients_average(
            global_model,
            client_train_loaders,
        )

        val_metrics = evaluate_clients_average(
            global_model,
            client_val_loaders,
        )

        print(
            f"FedDrop | Round {round_number:03d} | "
            f"Train Acc: {train_metrics['acc']:.4f} | "
            f"Val Acc: {val_metrics['acc']:.4f}"
        )

        history["round"].append(round_number)
        history["train_acc"].append(
            train_metrics["acc"]
        )
        history["val_acc"].append(
            val_metrics["acc"]
        )

        # Keep the best validation model
        if val_metrics["acc"] > best_val_acc + min_delta:
            best_val_acc = val_metrics["acc"]

            best_state = copy.deepcopy(
                global_model.state_dict()
            )

            best_round = round_number
            rounds_without_improvement = 0
        else:
            rounds_without_improvement += 1

        if (
            rounds_without_improvement
            >= early_stop_patience
        ):
            print(
                f"FedDrop early stopping at round "
                f"{round_number}. "
                f"Best round: {best_round}"
            )
            break

    global_model.load_state_dict(best_state)
    global_model.to(DEVICE)
    global_model.eval()

    history["best_round"] = best_round

    final_test_metrics = evaluate_loss_and_metrics(
        global_model,
        test_loader,
    )

    roadmap, roadmap_metadata = (
        build_feddrop_roadmap(global_model)
    )

    torch.save(
        {
            "model_state_dict": (
                global_model.state_dict()
            ),
            "submodel_roadmap": (
                convert_roadmap_to_cpu(roadmap)
            ),
            "submodel_roadmap_metadata": (
                roadmap_metadata
            ),
            "history": history,
            "final_test_metrics": (
                final_test_metrics
            ),
            "config": {
                "method": "FedDrop",
                "dataset": "CIFAR10",
                "architecture": "VGG-style",
                "remove_percentages": (
                    REMOVE_PERCENTAGES
                ),
                "min_layer_active_ratio": (
                    MIN_LAYER_ACTIVE_RATIO
                ),
                "uses_importance": False,
            },
        },
        save_path,
    )

    print(
        "FedDrop final test:",
        final_test_metrics,
    )
    print("Saved:", save_path)

    return (
        global_model,
        roadmap,
        roadmap_metadata,
        history,
        final_test_metrics,
    )


(
    feddrop_model,
    feddrop_roadmap,
    feddrop_metadata,
    feddrop_history,
    feddrop_test_metrics,
) = run_feddrop()

In [ ]:
# ============================================================
# CELL 7: OFA-Style Baseline Training
# Width-Supernet Training
# ============================================================

OFA_PATH = "OFA_Style_CIFAR10_VGG.pt"


def build_nested_width_mask_ofa(
    model,
    remove_percentage,
):
    structured_mask = {}
    final_layer = get_final_structurable_layer_name(model)

    total_prunable = 0
    total_removed = 0

    for name, module in get_structurable_layers(model):
        if isinstance(module, nn.Conv2d):
            size = module.out_channels
        else:
            size = module.out_features

        if (
            PROTECT_FINAL_CLASSIFIER
            and name == final_layer
        ):
            structured_mask[name] = torch.ones(
                size,
                device=DEVICE,
            )
            continue

        total_prunable += size

        keep_ratio = 1.0 - remove_percentage / 100.0
        requested_keep = int(
            math.ceil(keep_ratio * size)
        )

        minimum_keep = get_layer_min_count(
            model,
            name,
            module,
            size,
        )

        num_keep = min(
            size,
            max(minimum_keep, requested_keep),
        )

        mask = torch.zeros(
            size,
            device=DEVICE,
        )
        mask[:num_keep] = 1.0

        structured_mask[name] = mask
        total_removed += size - num_keep

    metadata = {
        "requested_remove_percentage": remove_percentage,
        "actual_removed": total_removed,
        "actual_removed_ratio": (
            total_removed / max(total_prunable, 1)
        ),
        "uses_importance": False,
        "uses_personalization": False,
        "mask_type": "ofa_nested_width",
    }

    return structured_mask, metadata


def build_ofa_roadmap(model):
    roadmap = {
        "full": build_full_structured_mask(model)
    }

    metadata = {
        "full": {
            "requested_remove_percentage": 0,
            "actual_removed_ratio": 0.0,
        }
    }

    for percentage in REMOVE_PERCENTAGES:
        mode = f"ofa_remove_{percentage}"

        roadmap[mode], metadata[mode] = (
            build_nested_width_mask_ofa(
                model,
                percentage,
            )
        )

    return roadmap, metadata


def train_one_client_ofa(
    local_model,
    loader,
    lr,
    epochs,
):
    local_model.train()

    optimizer = torch.optim.SGD(
        local_model.parameters(),
        lr=lr,
        momentum=0.9,
        weight_decay=5e-4,
    )

    criterion = nn.CrossEntropyLoss()

    for _ in range(epochs):
        for inputs, labels in loader:
            inputs = inputs.to(DEVICE)
            labels = labels.to(DEVICE)

            # Train the full model first
            optimizer.zero_grad(set_to_none=True)

            logits = local_model(inputs)
            full_loss = criterion(logits, labels)

            full_loss.backward()
            optimizer.step()

            # Train one smaller width using the same batch
            remove_percentage = random.choice(
                REMOVE_PERCENTAGES
            )

            structured_mask, _ = (
                build_nested_width_mask_ofa(
                    local_model,
                    remove_percentage,
                )
            )

            parameter_masks = (
                expand_structured_masks_to_parameter_masks(
                    local_model,
                    structured_mask,
                )
            )

            parameter_backup = {
                name: parameter.data.clone()
                for name, parameter
                in local_model.named_parameters()
                if name in parameter_masks
            }

            optimizer.zero_grad(set_to_none=True)

            apply_parameter_masks_to_model(
                local_model,
                parameter_masks,
            )

            submodel_logits = local_model(inputs)
            submodel_loss = criterion(
                submodel_logits,
                labels,
            )

            submodel_loss.backward()

            # Restore the full model before updating it
            with torch.no_grad():
                for name, parameter in (
                    local_model.named_parameters()
                ):
                    if name in parameter_backup:
                        parameter.data.copy_(
                            parameter_backup[name]
                        )

            mask_gradients(
                local_model,
                parameter_masks,
            )

            optimizer.step()

    local_model.eval()
    return local_model


def run_ofa(
    rounds=ROUNDS,
    local_epochs=LOCAL_EPOCHS,
    lr=LR,
    early_stop_patience=5,
    min_delta=1e-4,
    save_path=OFA_PATH,
):
    global_model = new_model()

    history = {
        "round": [],
        "train_acc": [],
        "val_acc": [],
        "best_round": None,
    }

    best_val_acc = -float("inf")
    best_state = copy.deepcopy(
        global_model.state_dict()
    )
    best_round = 0
    rounds_without_improvement = 0

    for round_number in range(1, rounds + 1):
        local_models = []
        local_weights = []

        # Train each client from the global model
        for client_id in range(NUM_CLIENTS):
            local_model = clone_model(global_model)

            local_model = train_one_client_ofa(
                local_model=local_model,
                loader=client_train_loaders[client_id],
                lr=lr,
                epochs=local_epochs,
            )

            local_models.append(local_model)
            local_weights.append(
                len(client_train_indices[client_id])
            )

        global_model = average_models(
            local_models,
            local_weights,
        )

        train_metrics = evaluate_clients_average(
            global_model,
            client_train_loaders,
        )

        val_metrics = evaluate_clients_average(
            global_model,
            client_val_loaders,
        )

        print(
            f"OFA | Round {round_number:03d} | "
            f"Train Acc: {train_metrics['acc']:.4f} | "
            f"Val Acc: {val_metrics['acc']:.4f}"
        )

        history["round"].append(round_number)
        history["train_acc"].append(
            train_metrics["acc"]
        )
        history["val_acc"].append(
            val_metrics["acc"]
        )

        # Keep the best validation model
        if val_metrics["acc"] > best_val_acc + min_delta:
            best_val_acc = val_metrics["acc"]

            best_state = copy.deepcopy(
                global_model.state_dict()
            )

            best_round = round_number
            rounds_without_improvement = 0
        else:
            rounds_without_improvement += 1

        if (
            rounds_without_improvement
            >= early_stop_patience
        ):
            print(
                f"OFA early stopping at round "
                f"{round_number}. "
                f"Best round: {best_round}"
            )
            break

    # Load the best round before testing
    global_model.load_state_dict(best_state)
    global_model.to(DEVICE)
    global_model.eval()

    history["best_round"] = best_round

    final_test_metrics = evaluate_loss_and_metrics(
        global_model,
        test_loader,
    )

    roadmap, roadmap_metadata = (
        build_ofa_roadmap(global_model)
    )

    torch.save(
        {
            "model_state_dict": (
                global_model.state_dict()
            ),
            "submodel_roadmap": (
                convert_roadmap_to_cpu(roadmap)
            ),
            "submodel_roadmap_metadata": (
                roadmap_metadata
            ),
            "history": history,
            "final_test_metrics": (
                final_test_metrics
            ),
            "config": {
                "method": "OFA",
                "dataset": "CIFAR10",
                "architecture": "VGG-style",
                "remove_percentages": (
                    REMOVE_PERCENTAGES
                ),
                "min_layer_active_ratio": (
                    MIN_LAYER_ACTIVE_RATIO
                ),
            },
        },
        save_path,
    )

    print(
        "OFA final test:",
        final_test_metrics,
    )
    print("Saved:", save_path)

    return (
        global_model,
        roadmap,
        roadmap_metadata,
        history,
        final_test_metrics,
    )


(
    ofa_model,
    ofa_roadmap,
    ofa_metadata,
    ofa_history,
    ofa_test_metrics,
) = run_ofa()

In [ ]:
# ============================================================
# CELL 8: SLEXNet Baseline Training
# Width Scaling and Early Exits
# ============================================================

SLEX_PATH = "SLEXNet_Style_CIFAR10_VGG.pt"


class SLEXVGGSmall(nn.Module):
    def __init__(self, num_classes=NUM_CLASSES):
        super().__init__()

        self.block1 = nn.Sequential(
            nn.Conv2d(3, 64, 3, padding=1, bias=False),
            nn.BatchNorm2d(64),
            nn.ReLU(inplace=True),
            nn.Conv2d(64, 64, 3, padding=1, bias=False),
            nn.BatchNorm2d(64),
            nn.ReLU(inplace=True),
            nn.MaxPool2d(2, 2),
            nn.Dropout2d(0.10),
        )

        self.block2 = nn.Sequential(
            nn.Conv2d(64, 128, 3, padding=1, bias=False),
            nn.BatchNorm2d(128),
            nn.ReLU(inplace=True),
            nn.Conv2d(128, 128, 3, padding=1, bias=False),
            nn.BatchNorm2d(128),
            nn.ReLU(inplace=True),
            nn.MaxPool2d(2, 2),
            nn.Dropout2d(0.15),
        )

        self.block3 = nn.Sequential(
            nn.Conv2d(128, 256, 3, padding=1, bias=False),
            nn.BatchNorm2d(256),
            nn.ReLU(inplace=True),
            nn.Conv2d(256, 256, 3, padding=1, bias=False),
            nn.BatchNorm2d(256),
            nn.ReLU(inplace=True),
            nn.MaxPool2d(2, 2),
            nn.Dropout2d(0.20),
        )

        self.exit1 = nn.Sequential(
            nn.AdaptiveAvgPool2d((1, 1)),
            nn.Flatten(),
            nn.Linear(64, num_classes),
        )

        self.exit2 = nn.Sequential(
            nn.AdaptiveAvgPool2d((1, 1)),
            nn.Flatten(),
            nn.Linear(128, num_classes),
        )

        self.exit3 = nn.Sequential(
            nn.AdaptiveAvgPool2d((1, 1)),
            nn.Flatten(),
            nn.Linear(256, num_classes),
        )

        self.classifier = nn.Sequential(
            nn.Flatten(),
            nn.Linear(256 * 4 * 4, 512, bias=False),
            nn.LayerNorm(512),
            nn.ReLU(inplace=True),
            nn.Dropout(0.50),
            nn.Linear(512, num_classes),
        )

    def forward_all(self, inputs):
        features1 = self.block1(inputs)
        exit1_logits = self.exit1(features1)

        features2 = self.block2(features1)
        exit2_logits = self.exit2(features2)

        features3 = self.block3(features2)
        exit3_logits = self.exit3(features3)

        final_logits = self.classifier(features3)

        return {
            "exit1": exit1_logits,
            "exit2": exit2_logits,
            "exit3": exit3_logits,
            "final": final_logits,
        }

    def forward(self, inputs, exit_name="final"):
        return self.forward_all(inputs)[exit_name]


def new_slex_model():
    return SLEXVGGSmall().to(DEVICE)


def clone_slex_model(model):
    cloned_model = copy.deepcopy(model)
    return cloned_model.to(DEVICE)


def average_slex_models(models, weights=None):
    if not models:
        raise ValueError("At least one model is required.")

    if weights is None:
        weights = np.ones(len(models), dtype=np.float64)
    else:
        weights = np.asarray(weights, dtype=np.float64)

    if len(weights) != len(models):
        raise ValueError("The number of weights must match the models.")

    if weights.sum() <= 0:
        raise ValueError("Model weights must have a positive sum.")

    weights /= weights.sum()

    averaged_model = clone_slex_model(models[0])
    averaged_state = copy.deepcopy(
        averaged_model.state_dict()
    )

    for name, value in averaged_state.items():
        if torch.is_floating_point(value):
            averaged_state[name] = torch.zeros_like(value)

    for model, weight in zip(models, weights):
        state = model.state_dict()

        for name, value in averaged_state.items():
            if torch.is_floating_point(value):
                averaged_state[name] += (
                    state[name].to(value.device)
                    * float(weight)
                )
            else:
                averaged_state[name] = state[name].clone()

    averaged_model.load_state_dict(averaged_state)
    averaged_model.to(DEVICE)
    averaged_model.eval()

    return averaged_model


def evaluate_slex_loss_and_metrics(
    model,
    loader,
    exit_name="final",
):
    model.eval()

    criterion = nn.CrossEntropyLoss(reduction="sum")

    total_loss = 0.0
    total_correct = 0
    total_samples = 0

    predictions = []
    targets = []

    with torch.no_grad():
        for inputs, labels in loader:
            inputs = inputs.to(DEVICE)
            labels = labels.to(DEVICE)

            logits = model(
                inputs,
                exit_name=exit_name,
            )

            loss = criterion(logits, labels)
            predicted_labels = logits.argmax(dim=1)

            total_loss += loss.item()
            total_correct += (
                predicted_labels == labels
            ).sum().item()
            total_samples += labels.size(0)

            predictions.extend(
                predicted_labels.cpu().tolist()
            )
            targets.extend(labels.cpu().tolist())

    accuracy = total_correct / max(total_samples, 1)
    average_loss = total_loss / max(total_samples, 1)

    if f1_score is None:
        macro_f1 = accuracy
        balanced_accuracy = accuracy
    else:
        macro_f1 = f1_score(
            targets,
            predictions,
            average="macro",
            zero_division=0,
        )

        balanced_accuracy = balanced_accuracy_score(
            targets,
            predictions,
        )

    return {
        "loss": average_loss,
        "acc": accuracy,
        "macro_f1": macro_f1,
        "balanced_acc": balanced_accuracy,
    }


def get_structurable_layers_slex(model):
    return [
        (name, module)
        for name, module in model.named_modules()
        if isinstance(module, (nn.Conv2d, nn.Linear))
    ]


def is_output_classifier_layer_slex(name, module):
    return (
        isinstance(module, nn.Linear)
        and module.out_features == NUM_CLASSES
    )


def build_full_structured_mask_slex(model):
    structured_mask = {}

    for name, module in get_structurable_layers_slex(model):
        if isinstance(module, nn.Conv2d):
            size = module.out_channels
        else:
            size = module.out_features

        structured_mask[name] = torch.ones(
            size,
            device=DEVICE,
        )

    return structured_mask


def build_nested_width_mask_slex(
    model,
    remove_percentage,
):
    structured_mask = {}

    total_prunable = 0
    total_removed = 0

    for name, module in get_structurable_layers_slex(model):
        if isinstance(module, nn.Conv2d):
            size = module.out_channels
        else:
            size = module.out_features

        if (
            PROTECT_FINAL_CLASSIFIER
            and is_output_classifier_layer_slex(
                name,
                module,
            )
        ):
            structured_mask[name] = torch.ones(
                size,
                device=DEVICE,
            )
            continue

        total_prunable += size

        keep_ratio = 1.0 - remove_percentage / 100.0
        requested_keep = int(
            math.ceil(keep_ratio * size)
        )

        minimum_keep = min(
            size,
            max(
                MIN_LAYER_ACTIVE_STRUCTURES,
                int(
                    math.ceil(
                        MIN_LAYER_ACTIVE_RATIO * size
                    )
                ),
            ),
        )

        num_keep = min(
            size,
            max(minimum_keep, requested_keep),
        )

        mask = torch.zeros(
            size,
            device=DEVICE,
        )
        mask[:num_keep] = 1.0

        structured_mask[name] = mask
        total_removed += size - num_keep

    metadata = {
        "requested_remove_percentage": remove_percentage,
        "actual_removed": total_removed,
        "actual_removed_ratio": (
            total_removed / max(total_prunable, 1)
        ),
        "mask_type": "slex_nested_width",
    }

    return structured_mask, metadata


def build_slex_roadmap(model):
    roadmap = {
        "full": build_full_structured_mask_slex(model)
    }

    metadata = {
        "full": {
            "requested_remove_percentage": 0,
            "actual_removed_ratio": 0.0,
        }
    }

    for percentage in REMOVE_PERCENTAGES:
        mode = f"slex_remove_{percentage}"

        roadmap[mode], metadata[mode] = (
            build_nested_width_mask_slex(
                model,
                percentage,
            )
        )

    return roadmap, metadata


def expand_structured_masks_to_parameter_masks_slex(
    model,
    structured_masks,
):
    parameter_masks = {}
    previous_feature_mask = None

    for layer_name, module in model.named_modules():
        if (
            isinstance(module, nn.Conv2d)
            and layer_name in structured_masks
        ):
            mask = structured_masks[layer_name].to(
                DEVICE
            )

            parameter_masks[
                f"{layer_name}.weight"
            ] = mask.view(
                -1,
                1,
                1,
                1,
            ).expand_as(module.weight.data)

            if module.bias is not None:
                parameter_masks[
                    f"{layer_name}.bias"
                ] = mask.view(-1).expand_as(
                    module.bias.data
                )

            previous_feature_mask = mask

        elif isinstance(module, nn.BatchNorm2d):
            if (
                previous_feature_mask is not None
                and module.num_features
                == previous_feature_mask.numel()
            ):
                mask = previous_feature_mask.to(
                    DEVICE
                )

                if module.weight is not None:
                    parameter_masks[
                        f"{layer_name}.weight"
                    ] = mask.view(-1).expand_as(
                        module.weight.data
                    )

                if module.bias is not None:
                    parameter_masks[
                        f"{layer_name}.bias"
                    ] = mask.view(-1).expand_as(
                        module.bias.data
                    )

        elif (
            isinstance(module, nn.Linear)
            and layer_name in structured_masks
        ):
            mask = structured_masks[layer_name].to(
                DEVICE
            )

            parameter_masks[
                f"{layer_name}.weight"
            ] = mask.view(
                -1,
                1,
            ).expand_as(module.weight.data)

            if module.bias is not None:
                parameter_masks[
                    f"{layer_name}.bias"
                ] = mask.view(-1).expand_as(
                    module.bias.data
                )

            previous_feature_mask = mask

        elif isinstance(module, nn.LayerNorm):
            if (
                previous_feature_mask is not None
                and len(module.normalized_shape) == 1
                and module.normalized_shape[0]
                == previous_feature_mask.numel()
            ):
                mask = previous_feature_mask.to(
                    DEVICE
                )

                if module.elementwise_affine:
                    parameter_masks[
                        f"{layer_name}.weight"
                    ] = mask.view(-1).expand_as(
                        module.weight.data
                    )

                    parameter_masks[
                        f"{layer_name}.bias"
                    ] = mask.view(-1).expand_as(
                        module.bias.data
                    )

    return parameter_masks


def apply_parameter_masks_to_model_slex(
    model,
    parameter_masks,
):
    with torch.no_grad():
        for name, parameter in model.named_parameters():
            if name in parameter_masks:
                parameter.data.mul_(
                    parameter_masks[name].to(
                        parameter.device
                    )
                )


def mask_gradients_slex(
    model,
    parameter_masks,
):
    with torch.no_grad():
        for name, parameter in model.named_parameters():
            if (
                name in parameter_masks
                and parameter.grad is not None
            ):
                parameter.grad.mul_(
                    parameter_masks[name].to(
                        parameter.device
                    )
                )


def count_active_structures_slex(structured_mask):
    active = 0
    total = 0

    for mask in structured_mask.values():
        active += int(mask.sum().item())
        total += int(mask.numel())

    active_ratio = active / max(total, 1)

    return active, total, active_ratio


def convert_slex_roadmap_to_cpu(roadmap):
    return {
        mode: {
            name: mask.detach().cpu()
            for name, mask in layer_masks.items()
        }
        for mode, layer_masks in roadmap.items()
    }


def move_slex_roadmap_to_device(
    roadmap,
    device=DEVICE,
):
    return {
        mode: {
            name: mask.to(device)
            for name, mask in layer_masks.items()
        }
        for mode, layer_masks in roadmap.items()
    }


def compute_slex_multi_exit_loss(
    outputs,
    labels,
):
    criterion = nn.CrossEntropyLoss()

    return (
        0.25 * criterion(outputs["exit1"], labels)
        + 0.35 * criterion(outputs["exit2"], labels)
        + 0.45 * criterion(outputs["exit3"], labels)
        + 1.00 * criterion(outputs["final"], labels)
    )


def train_one_client_slex(
    local_model,
    loader,
    lr,
    epochs,
):
    local_model.train()

    optimizer = torch.optim.SGD(
        local_model.parameters(),
        lr=lr,
        momentum=0.9,
        weight_decay=5e-4,
    )

    for _ in range(epochs):
        for inputs, labels in loader:
            inputs = inputs.to(DEVICE)
            labels = labels.to(DEVICE)

            # Train the full model and all exits
            optimizer.zero_grad(set_to_none=True)

            outputs = local_model.forward_all(inputs)

            full_loss = compute_slex_multi_exit_loss(
                outputs,
                labels,
            )

            full_loss.backward()
            optimizer.step()

            # Train one smaller width for the same batch
            remove_percentage = random.choice(
                REMOVE_PERCENTAGES
            )

            structured_mask, _ = (
                build_nested_width_mask_slex(
                    local_model,
                    remove_percentage,
                )
            )

            parameter_masks = (
                expand_structured_masks_to_parameter_masks_slex(
                    local_model,
                    structured_mask,
                )
            )

            parameter_backup = {
                name: parameter.data.clone()
                for name, parameter
                in local_model.named_parameters()
                if name in parameter_masks
            }

            optimizer.zero_grad(set_to_none=True)

            apply_parameter_masks_to_model_slex(
                local_model,
                parameter_masks,
            )

            submodel_outputs = local_model.forward_all(
                inputs
            )

            submodel_loss = compute_slex_multi_exit_loss(
                submodel_outputs,
                labels,
            )

            submodel_loss.backward()

            # Restore the full model before updating it
            with torch.no_grad():
                for name, parameter in (
                    local_model.named_parameters()
                ):
                    if name in parameter_backup:
                        parameter.data.copy_(
                            parameter_backup[name]
                        )

            mask_gradients_slex(
                local_model,
                parameter_masks,
            )

            optimizer.step()

    local_model.eval()
    return local_model


def run_slex(
    rounds=ROUNDS,
    local_epochs=LOCAL_EPOCHS,
    lr=LR,
    early_stop_patience=5,
    min_delta=1e-4,
    save_path=SLEX_PATH,
):
    global_model = new_slex_model()

    history = {
        "round": [],
        "train_acc": [],
        "val_acc": [],
        "best_round": None,
    }

    best_val_acc = -float("inf")
    best_state = copy.deepcopy(
        global_model.state_dict()
    )
    best_round = 0
    rounds_without_improvement = 0

    for round_number in range(1, rounds + 1):
        local_models = []
        local_weights = []

        # Train each client from the current global model
        for client_id in range(NUM_CLIENTS):
            local_model = clone_slex_model(
                global_model
            )

            local_model = train_one_client_slex(
                local_model=local_model,
                loader=client_train_loaders[client_id],
                lr=lr,
                epochs=local_epochs,
            )

            local_models.append(local_model)

            local_weights.append(
                len(client_train_indices[client_id])
            )

        global_model = average_slex_models(
            local_models,
            local_weights,
        )

        train_accuracy = float(
            np.mean(
                [
                    evaluate_slex_loss_and_metrics(
                        global_model,
                        loader,
                        exit_name="final",
                    )["acc"]
                    for loader in client_train_loaders
                ]
            )
        )

        val_accuracy = float(
            np.mean(
                [
                    evaluate_slex_loss_and_metrics(
                        global_model,
                        loader,
                        exit_name="final",
                    )["acc"]
                    for loader in client_val_loaders
                ]
            )
        )

        print(
            f"SLEXNet | Round {round_number:03d} | "
            f"Train Acc: {train_accuracy:.4f} | "
            f"Val Acc: {val_accuracy:.4f}"
        )

        history["round"].append(round_number)
        history["train_acc"].append(train_accuracy)
        history["val_acc"].append(val_accuracy)

        # Keep the best validation model
        if val_accuracy > best_val_acc + min_delta:
            best_val_acc = val_accuracy

            best_state = copy.deepcopy(
                global_model.state_dict()
            )

            best_round = round_number
            rounds_without_improvement = 0
        else:
            rounds_without_improvement += 1

        if (
            rounds_without_improvement
            >= early_stop_patience
        ):
            print(
                f"SLEXNet early stopping at round "
                f"{round_number}. "
                f"Best round: {best_round}"
            )
            break

    # Load the best round before testing
    global_model.load_state_dict(best_state)
    global_model.to(DEVICE)
    global_model.eval()

    history["best_round"] = best_round

    final_test_metrics = (
        evaluate_slex_loss_and_metrics(
            global_model,
            test_loader,
            exit_name="final",
        )
    )

    roadmap, roadmap_metadata = (
        build_slex_roadmap(global_model)
    )

    torch.save(
        {
            "model_state_dict": (
                global_model.state_dict()
            ),
            "submodel_roadmap": (
                convert_slex_roadmap_to_cpu(
                    roadmap
                )
            ),
            "submodel_roadmap_metadata": (
                roadmap_metadata
            ),
            "history": history,
            "final_test_metrics": (
                final_test_metrics
            ),
            "config": {
                "method": "SLEXNet",
                "dataset": "CIFAR10",
                "architecture": "VGG-style",
                "remove_percentages": (
                    REMOVE_PERCENTAGES
                ),
                "min_layer_active_ratio": (
                    MIN_LAYER_ACTIVE_RATIO
                ),
                "exits": [
                    "exit1",
                    "exit2",
                    "exit3",
                    "final",
                ],
            },
        },
        save_path,
    )

    print(
        "SLEXNet final test:",
        final_test_metrics,
    )
    print("Saved:", save_path)

    return (
        global_model,
        roadmap,
        roadmap_metadata,
        history,
        final_test_metrics,
    )


(
    slex_model,
    slex_roadmap,
    slex_metadata,
    slex_history,
    slex_test_metrics,
) = run_slex()

In [ ]:
# ============================================================
# CELL 9: Slimmable Neural Network Baseline Training
# Standard Sandwich-Rule Width Training
# ============================================================

import copy
import numpy as np
import pandas as pd

import torch
import torch.nn as nn
import torch.nn.functional as F

try:
    from sklearn.metrics import f1_score, balanced_accuracy_score
except Exception:
    f1_score = None
    balanced_accuracy_score = None


SLIMMABLE_PATH = "Slimmable_CIFAR10_VGG.pt"

# Widths available during training and evaluation
REMOVE_PERCENTAGES = globals().get(
    "REMOVE_PERCENTAGES",
    [5, 10, 15, 20, 25, 30, 40, 50]
)


REMOVE_PERCENTAGES = sorted(
    list(set(list(REMOVE_PERCENTAGES) + [5, 10, 15, 20, 25, 30, 40, 50]))
)

MIN_LAYER_ACTIVE_RATIO = globals().get(
    "MIN_LAYER_ACTIVE_RATIO",
    0.50
)


SLIMMABLE_WIDTHS = [1.00] + [
    max(1.0 - p / 100.0, MIN_LAYER_ACTIVE_RATIO)
    for p in REMOVE_PERCENTAGES
]


SLIMMABLE_WIDTHS = list(
    dict.fromkeys([round(float(w), 2) for w in SLIMMABLE_WIDTHS])
)


SLIMMABLE_NUM_SAMPLED_WIDTHS = 1

SLIMMABLE_LR = LR
SLIMMABLE_WEIGHT_DECAY = 5e-4
SLIMMABLE_LOCAL_EPOCHS = LOCAL_EPOCHS
SLIMMABLE_MAX_ROUNDS = ROUNDS


SLIMMABLE_PATIENCE = globals().get("PATIENCE", 5)
SLIMMABLE_MIN_DELTA = 5e-4

print("Slimmable baseline: Standard Slimmable Neural Networks")
print("Slimmable widths:", SLIMMABLE_WIDTHS)
print("Sandwich rule: largest + smallest + one random intermediate width")
print("No distillation, no personalization, no compact-width checkpoint selection")
print("Slimmable max rounds:", SLIMMABLE_MAX_ROUNDS)
print("Slimmable patience:", SLIMMABLE_PATIENCE)
print("Slimmable min delta:", SLIMMABLE_MIN_DELTA)


class SlimmableConv2d(nn.Module):
    def __init__(
        self,
        in_channels,
        out_channels,
        kernel_size,
        stride=1,
        padding=0,
        bias=False
    ):
        super().__init__()

        self.in_channels_max = int(in_channels)
        self.out_channels_max = int(out_channels)
        self.active_width = 1.0

        self.conv = nn.Conv2d(
            in_channels,
            out_channels,
            kernel_size=kernel_size,
            stride=stride,
            padding=padding,
            bias=bias
        )

    def set_active_width(self, width):
        self.active_width = float(width)

    def forward(self, x):
        active_out = max(
            1,
            int(round(self.out_channels_max * self.active_width))
        )

        active_in = x.shape[1]

        weight = self.conv.weight[:active_out, :active_in, :, :]

        if self.conv.bias is not None:
            bias = self.conv.bias[:active_out]
        else:
            bias = None

        return F.conv2d(
            x,
            weight,
            bias,
            stride=self.conv.stride,
            padding=self.conv.padding
        )


class SlimmableBatchNorm2d(nn.Module):
    def __init__(self, num_features):
        super().__init__()

        self.num_features_max = int(num_features)
        self.active_width = 1.0

        self.bn = nn.BatchNorm2d(num_features)

    def set_active_width(self, width):
        self.active_width = float(width)

    def forward(self, x):
        active_features = x.shape[1]

        return F.batch_norm(
            x,
            self.bn.running_mean[:active_features],
            self.bn.running_var[:active_features],
            self.bn.weight[:active_features],
            self.bn.bias[:active_features],
            training=self.bn.training,
            momentum=self.bn.momentum,
            eps=self.bn.eps
        )


class SlimmableLinear(nn.Module):
    def __init__(
        self,
        in_features,
        out_features,
        bias=False,
        is_classifier=False
    ):
        super().__init__()

        self.in_features_max = int(in_features)
        self.out_features_max = int(out_features)
        self.active_width = 1.0
        self.is_classifier = bool(is_classifier)

        self.linear = nn.Linear(
            in_features,
            out_features,
            bias=bias
        )

    def set_active_width(self, width):
        self.active_width = float(width)

    def forward(self, x):
        active_in = x.shape[1]

        # The classifier always keeps all output classes
        if self.is_classifier:

            active_out = self.out_features_max
        else:
            active_out = max(
                1,
                int(round(self.out_features_max * self.active_width))
            )

        weight = self.linear.weight[:active_out, :active_in]

        if self.linear.bias is not None:
            bias = self.linear.bias[:active_out]
        else:
            bias = None

        return F.linear(
            x,
            weight,
            bias
        )


class SlimmableLayerNorm(nn.Module):
    def __init__(self, normalized_shape):
        super().__init__()

        self.normalized_shape_max = int(normalized_shape)
        self.active_width = 1.0

        self.ln = nn.LayerNorm(normalized_shape)

    def set_active_width(self, width):
        self.active_width = float(width)

    def forward(self, x):
        active_features = x.shape[1]

        return F.layer_norm(
            x,
            normalized_shape=(active_features,),
            weight=self.ln.weight[:active_features],
            bias=self.ln.bias[:active_features],
            eps=self.ln.eps
        )


class SlimmableCIFAR10VGGSmall(nn.Module):


    def __init__(self, num_classes=NUM_CLASSES):
        super().__init__()

        self.active_width = 1.0

        self.conv1 = SlimmableConv2d(
            3,
            64,
            kernel_size=3,
            padding=1,
            bias=False
        )
        self.bn1 = SlimmableBatchNorm2d(64)

        self.conv2 = SlimmableConv2d(
            64,
            64,
            kernel_size=3,
            padding=1,
            bias=False
        )
        self.bn2 = SlimmableBatchNorm2d(64)

        self.conv3 = SlimmableConv2d(
            64,
            128,
            kernel_size=3,
            padding=1,
            bias=False
        )
        self.bn3 = SlimmableBatchNorm2d(128)

        self.conv4 = SlimmableConv2d(
            128,
            128,
            kernel_size=3,
            padding=1,
            bias=False
        )
        self.bn4 = SlimmableBatchNorm2d(128)

        self.conv5 = SlimmableConv2d(
            128,
            256,
            kernel_size=3,
            padding=1,
            bias=False
        )
        self.bn5 = SlimmableBatchNorm2d(256)

        self.conv6 = SlimmableConv2d(
            256,
            256,
            kernel_size=3,
            padding=1,
            bias=False
        )
        self.bn6 = SlimmableBatchNorm2d(256)

        self.pool = nn.MaxPool2d(2, 2)

        self.drop1 = nn.Dropout2d(0.10)
        self.drop2 = nn.Dropout2d(0.15)
        self.drop3 = nn.Dropout2d(0.20)
        self.drop_fc = nn.Dropout(0.50)


        self.fc1 = SlimmableLinear(
            256 * 4 * 4,
            512,
            bias=False,
            is_classifier=False
        )

        self.ln1 = SlimmableLayerNorm(512)

        self.classifier = SlimmableLinear(
            512,
            num_classes,
            bias=True,
            is_classifier=True
        )

    def set_active_width(self, width):
        self.active_width = float(width)

        for module in self.modules():
            if module is not self and hasattr(module, "set_active_width"):
                module.set_active_width(width)

    def forward(self, x):
        x = F.relu(self.bn1(self.conv1(x)), inplace=True)
        x = F.relu(self.bn2(self.conv2(x)), inplace=True)
        x = self.pool(x)
        x = self.drop1(x)

        x = F.relu(self.bn3(self.conv3(x)), inplace=True)
        x = F.relu(self.bn4(self.conv4(x)), inplace=True)
        x = self.pool(x)
        x = self.drop2(x)

        x = F.relu(self.bn5(self.conv5(x)), inplace=True)
        x = F.relu(self.bn6(self.conv6(x)), inplace=True)
        x = self.pool(x)
        x = self.drop3(x)

        x = torch.flatten(x, 1)

        x = F.relu(self.ln1(self.fc1(x)), inplace=True)
        x = self.drop_fc(x)

        x = self.classifier(x)

        return x


def new_slimmable_model():
    return SlimmableCIFAR10VGGSmall(
        num_classes=NUM_CLASSES
    ).to(DEVICE)


def clone_slimmable_model(model):
    cloned = copy.deepcopy(model)
    cloned.to(DEVICE)
    return cloned


def count_stored_slimmable_parameters(model):


    return int(
        sum(p.numel() for p in model.parameters())
    )


print(
    "Stored Slimmable supernet parameters:",
    count_stored_slimmable_parameters(new_slimmable_model())
)


def _conv2d_flops(
    in_channels,
    out_channels,
    kernel_size,
    out_h,
    out_w,
    bias=False
):


    kh, kw = kernel_size

    flops = (
        out_h
        * out_w
        * out_channels
        * in_channels
        * kh
        * kw
    )

    if bias:
        flops += out_h * out_w * out_channels

    return int(flops)


def _linear_flops(
    in_features,
    out_features,
    bias=False
):
    flops = int(in_features * out_features)

    if bias:
        flops += int(out_features)

    return int(flops)


def count_slimmable_active_parameters(model, width):


    width = float(width)

    c1 = max(1, int(round(model.conv1.out_channels_max * width)))
    c2 = max(1, int(round(model.conv2.out_channels_max * width)))
    c3 = max(1, int(round(model.conv3.out_channels_max * width)))
    c4 = max(1, int(round(model.conv4.out_channels_max * width)))
    c5 = max(1, int(round(model.conv5.out_channels_max * width)))
    c6 = max(1, int(round(model.conv6.out_channels_max * width)))
    h1 = max(1, int(round(model.fc1.out_features_max * width)))

    params = 0


    params += c1 * 3 * 3 * 3
    if model.conv1.conv.bias is not None:
        params += c1
    params += 2 * c1


    params += c2 * c1 * 3 * 3
    if model.conv2.conv.bias is not None:
        params += c2
    params += 2 * c2


    params += c3 * c2 * 3 * 3
    if model.conv3.conv.bias is not None:
        params += c3
    params += 2 * c3


    params += c4 * c3 * 3 * 3
    if model.conv4.conv.bias is not None:
        params += c4
    params += 2 * c4


    params += c5 * c4 * 3 * 3
    if model.conv5.conv.bias is not None:
        params += c5
    params += 2 * c5


    params += c6 * c5 * 3 * 3
    if model.conv6.conv.bias is not None:
        params += c6
    params += 2 * c6


    fc1_in = c6 * 4 * 4
    params += h1 * fc1_in
    if model.fc1.linear.bias is not None:
        params += h1


    params += 2 * h1


    params += model.classifier.out_features_max * h1
    if model.classifier.linear.bias is not None:
        params += model.classifier.out_features_max

    return int(params)


def count_slimmable_active_flops(model, width):


    width = float(width)

    c1 = max(1, int(round(model.conv1.out_channels_max * width)))
    c2 = max(1, int(round(model.conv2.out_channels_max * width)))
    c3 = max(1, int(round(model.conv3.out_channels_max * width)))
    c4 = max(1, int(round(model.conv4.out_channels_max * width)))
    c5 = max(1, int(round(model.conv5.out_channels_max * width)))
    c6 = max(1, int(round(model.conv6.out_channels_max * width)))
    h1 = max(1, int(round(model.fc1.out_features_max * width)))

    flops = 0


    flops += _conv2d_flops(
        in_channels=3,
        out_channels=c1,
        kernel_size=(3, 3),
        out_h=32,
        out_w=32,
        bias=model.conv1.conv.bias is not None
    )

    flops += _conv2d_flops(
        in_channels=c1,
        out_channels=c2,
        kernel_size=(3, 3),
        out_h=32,
        out_w=32,
        bias=model.conv2.conv.bias is not None
    )


    flops += _conv2d_flops(
        in_channels=c2,
        out_channels=c3,
        kernel_size=(3, 3),
        out_h=16,
        out_w=16,
        bias=model.conv3.conv.bias is not None
    )

    flops += _conv2d_flops(
        in_channels=c3,
        out_channels=c4,
        kernel_size=(3, 3),
        out_h=16,
        out_w=16,
        bias=model.conv4.conv.bias is not None
    )


    flops += _conv2d_flops(
        in_channels=c4,
        out_channels=c5,
        kernel_size=(3, 3),
        out_h=8,
        out_w=8,
        bias=model.conv5.conv.bias is not None
    )

    flops += _conv2d_flops(
        in_channels=c5,
        out_channels=c6,
        kernel_size=(3, 3),
        out_h=8,
        out_w=8,
        bias=model.conv6.conv.bias is not None
    )


    fc1_in = c6 * 4 * 4
    flops += _linear_flops(
        in_features=fc1_in,
        out_features=h1,
        bias=model.fc1.linear.bias is not None
    )


    flops += _linear_flops(
        in_features=h1,
        out_features=model.classifier.out_features_max,
        bias=model.classifier.linear.bias is not None
    )

    return int(flops)


def build_slimmable_active_runtime_profile(model):
    rows = []

    for width in SLIMMABLE_WIDTHS:
        remove_percent = int(round((1.0 - width) * 100))

        active_params = count_slimmable_active_parameters(
            model,
            width
        )

        active_flops = count_slimmable_active_flops(
            model,
            width
        )

        rows.append({
            "width": width,
            "remove_low_percent": remove_percent,
            "active_params": active_params,
            "active_flops": active_flops,
            "params": active_params,
            "flops": active_flops,
        })

    return pd.DataFrame(rows)


def sample_slimmable_widths(width_list, num_sampled=1):
    # Use the largest, smallest, and one random middle width

    width_list = sorted(
        [float(w) for w in width_list],
        reverse=True
    )

    largest = max(width_list)
    smallest = min(width_list)

    middle = [
        w for w in width_list
        if w not in [largest, smallest]
    ]

    if len(middle) == 0 or num_sampled <= 0:
        return [largest, smallest]

    sampled = list(
        np.random.choice(
            middle,
            size=min(num_sampled, len(middle)),
            replace=False
        )
    )

    return [largest] + sampled + [smallest]


def train_slimmable_supervised(
    model,
    loader,
    lr=SLIMMABLE_LR,
    epochs=SLIMMABLE_LOCAL_EPOCHS
):


    model.train()

    optimizer = torch.optim.SGD(
        model.parameters(),
        lr=lr,
        momentum=0.9,
        weight_decay=SLIMMABLE_WEIGHT_DECAY
    )

    criterion = nn.CrossEntropyLoss()

    total_correct = 0
    total_examples = 0
    total_loss = 0.0
    total_batches = 0

    for _ in range(epochs):
        for x, y in loader:
            x = x.to(DEVICE)
            y = y.to(DEVICE)

            optimizer.zero_grad(set_to_none=True)

            sampled_widths = sample_slimmable_widths(
                SLIMMABLE_WIDTHS,
                num_sampled=SLIMMABLE_NUM_SAMPLED_WIDTHS
            )

            loss_sum = 0.0

            for width in sampled_widths:
                model.set_active_width(width)

                logits = model(x)
                loss = criterion(logits, y)

                loss_sum = loss_sum + loss

                with torch.no_grad():
                    preds = logits.argmax(dim=1)
                    total_correct += (preds == y).sum().item()
                    total_examples += y.size(0)


            # Update once using the average loss across sampled widths
            loss_avg = loss_sum / max(len(sampled_widths), 1)

            loss_avg.backward()
            optimizer.step()

            total_loss += loss_avg.item()
            total_batches += 1

    avg_loss = total_loss / max(total_batches, 1)
    avg_acc = total_correct / max(total_examples, 1)

    model.eval()

    return model, avg_loss, avg_acc


def average_slimmable_models(models, weights=None):
    if weights is None:
        weights = [1.0 for _ in models]

    weights = np.array(weights, dtype=np.float64)
    weights = weights / max(weights.sum(), 1e-12)

    avg_model = clone_slimmable_model(models[0])
    avg_state = copy.deepcopy(avg_model.state_dict())

    for key in avg_state:
        if torch.is_floating_point(avg_state[key]):
            avg_state[key] = torch.zeros_like(avg_state[key])

    for model, weight in zip(models, weights):
        state = model.state_dict()

        for key in avg_state:
            if torch.is_floating_point(avg_state[key]):
                avg_state[key] += (
                    state[key].to(avg_state[key].device)
                    * float(weight)
                )
            else:
                avg_state[key] = state[key].clone()

    avg_model.load_state_dict(avg_state)
    avg_model.to(DEVICE)
    avg_model.eval()

    return avg_model


def evaluate_slimmable_model(model, loader, width=1.0):
    model.eval()
    model.set_active_width(width)

    criterion = nn.CrossEntropyLoss(reduction="sum")

    loss_sum = 0.0
    correct = 0
    total = 0

    preds_all = []
    y_all = []

    with torch.no_grad():
        for x, y in loader:
            x = x.to(DEVICE)
            y = y.to(DEVICE)

            logits = model(x)
            loss = criterion(logits, y)
            pred = logits.argmax(dim=1)

            loss_sum += loss.item()
            correct += (pred == y).sum().item()
            total += y.numel()

            preds_all += pred.cpu().tolist()
            y_all += y.cpu().tolist()

    acc = correct / max(total, 1)
    loss = loss_sum / max(total, 1)

    if f1_score is not None:
        macro_f1 = f1_score(
            y_all,
            preds_all,
            average="macro",
            zero_division=0
        )

        balanced_acc = balanced_accuracy_score(
            y_all,
            preds_all
        )

    else:
        macro_f1 = acc
        balanced_acc = acc

    return {
        "loss": loss,
        "acc": acc,
        "macro_f1": macro_f1,
        "balanced_acc": balanced_acc,
    }


def evaluate_slimmable_clients_average(model, loaders, width=1.0):
    metrics_list = [
        evaluate_slimmable_model(
            model,
            loader,
            width=width
        )
        for loader in loaders
    ]

    return {
        k: float(np.mean([m[k] for m in metrics_list]))
        for k in metrics_list[0]
    }


def evaluate_slimmable_all_widths(model, loader):
    rows = []

    runtime_profile = build_slimmable_active_runtime_profile(model)

    runtime_by_width = {
        float(row["width"]): row
        for _, row in runtime_profile.iterrows()
    }

    for width in SLIMMABLE_WIDTHS:
        metrics = evaluate_slimmable_model(
            model,
            loader,
            width=width
        )

        active_params = int(
            runtime_by_width[float(width)]["active_params"]
        )

        active_flops = int(
            runtime_by_width[float(width)]["active_flops"]
        )

        rows.append({
            "width": width,
            "remove_low_percent": int(round((1.0 - width) * 100)),
            "active_params": active_params,
            "active_flops": active_flops,
            "params": active_params,
            "flops": active_flops,
            "loss": metrics["loss"],
            "acc": metrics["acc"],
            "macro_f1": metrics["macro_f1"],
            "balanced_acc": metrics["balanced_acc"],
        })

    return pd.DataFrame(rows)


def run_slimmable_fedavg(
    rounds=SLIMMABLE_MAX_ROUNDS,
    local_epochs=SLIMMABLE_LOCAL_EPOCHS,
    lr=SLIMMABLE_LR,
    client_fraction=1.0,
    patience=SLIMMABLE_PATIENCE,
    min_delta=SLIMMABLE_MIN_DELTA,
    save_path=SLIMMABLE_PATH
):


    global_model = new_slimmable_model()

    hist = {
        "round": [],
        "train_acc": [],
        "train_loss": [],
        "full_val_acc": [],
        "score": [],
        "best_round": None,
    }

    best_score = -float("inf")
    best_state = copy.deepcopy(global_model.state_dict())
    best_round = 0
    no_improve = 0

    for r in range(1, rounds + 1):
        selected_clients = np.random.choice(
            NUM_CLIENTS,
            size=max(1, int(client_fraction * NUM_CLIENTS)),
            replace=False
        )

        local_models = []
        local_weights = []
        local_losses = []
        local_accs = []

        for cid in selected_clients:
            local_model = clone_slimmable_model(global_model)

            local_model, local_loss, local_acc = train_slimmable_supervised(
                model=local_model,
                loader=client_train_loaders[cid],
                lr=lr,
                epochs=local_epochs
            )

            local_models.append(local_model)

            if "client_train_indices" in globals():
                local_weights.append(len(client_train_indices[cid]))
            else:
                local_weights.append(len(client_train_loaders[cid].dataset))

            local_losses.append(local_loss)
            local_accs.append(local_acc)

        global_model = average_slimmable_models(
            local_models,
            local_weights
        )


        # Choose the checkpoint using full-width validation
        full_val = evaluate_slimmable_clients_average(
            global_model,
            client_val_loaders,
            width=1.00
        )

        score = full_val["acc"]

        hist["round"].append(r)
        hist["train_acc"].append(float(np.mean(local_accs)))
        hist["train_loss"].append(float(np.mean(local_losses)))
        hist["full_val_acc"].append(full_val["acc"])
        hist["score"].append(score)

        print(
            f"Slimmable | Round {r:03d} | "
            f"Train Acc={np.mean(local_accs):.4f} | "
            f"Full Val={full_val['acc']:.4f} | "
            f"Checkpoint Score={score:.4f}"
        )

        if score > best_score + min_delta:
            best_score = score
            best_state = copy.deepcopy(global_model.state_dict())
            best_round = r
            no_improve = 0

        else:
            no_improve += 1

        if no_improve >= patience:
            print(
                f"Slimmable early stopping at round {r}. "
                f"Best round={best_round}"
            )
            break

    global_model.load_state_dict(best_state)
    global_model.to(DEVICE)
    global_model.eval()

    active_runtime_profile = build_slimmable_active_runtime_profile(
        global_model
    )

    width_results = evaluate_slimmable_all_widths(
        global_model,
        test_loader
    )

    full_test = evaluate_slimmable_model(
        global_model,
        test_loader,
        width=1.00
    )

    active_params_by_width = {
        str(float(row["width"])): int(row["active_params"])
        for _, row in active_runtime_profile.iterrows()
    }

    active_flops_by_width = {
        str(float(row["width"])): int(row["active_flops"])
        for _, row in active_runtime_profile.iterrows()
    }

    active_params_by_remove_percent = {
        f"remove_{int(row['remove_low_percent'])}": int(row["active_params"])
        for _, row in active_runtime_profile.iterrows()
    }

    active_flops_by_remove_percent = {
        f"remove_{int(row['remove_low_percent'])}": int(row["active_flops"])
        for _, row in active_runtime_profile.iterrows()
    }

    hist["best_round"] = best_round
    hist["best_score"] = best_score
    hist["test_acc"] = full_test["acc"]
    hist["test_loss"] = full_test["loss"]
    hist["test_macro_f1"] = full_test["macro_f1"]
    hist["test_balanced_acc"] = full_test["balanced_acc"]

    print("\nSlimmable ACTIVE runtime profile:")
    display(active_runtime_profile)

    print("\nSlimmable final test results by width:")
    display(width_results)

    print("\nSlimmable full-width final test:")
    print(full_test)

    # Save active subnet costs for the final comparison
    torch.save(
        {
            "model_state_dict": global_model.state_dict(),


            "width_list": SLIMMABLE_WIDTHS,
            "remove_percentages": REMOVE_PERCENTAGES,
            "min_layer_active_ratio": MIN_LAYER_ACTIVE_RATIO,


            "stored_supernet_params": count_stored_slimmable_parameters(global_model),
            "active_runtime_profile": active_runtime_profile,
            "active_params_by_width": active_params_by_width,
            "active_flops_by_width": active_flops_by_width,
            "active_params_by_remove_percent": active_params_by_remove_percent,
            "active_flops_by_remove_percent": active_flops_by_remove_percent,


            "history": hist,
            "test_width_results": width_results,
            "final_test_metrics": full_test,
            "best_round": best_round,
            "best_score": best_score,


            "config": {
                "method": "Slimmable Neural Networks",
                "dataset": "CIFAR-10",
                "architecture": "VGG-style",
                "noniid_alpha": NONIID_ALPHA,
                "num_classes": NUM_CLASSES,
                "rounds": rounds,
                "local_epochs": local_epochs,
                "lr": lr,
                "weight_decay": SLIMMABLE_WEIGHT_DECAY,
                "remove_percentages": REMOVE_PERCENTAGES,
                "width_rule": "active_width = max(1 - remove_low_percent / 100, MIN_LAYER_ACTIVE_RATIO)",


                "uses_standard_sandwich_rule": True,
                "sandwich_rule": "largest_width + smallest_width + one_random_intermediate_width",
                "num_sampled_intermediate_widths": SLIMMABLE_NUM_SAMPLED_WIDTHS,
                "uses_distillation": False,
                "uses_inplace_distillation": False,
                "uses_personalization": False,
                "uses_importance_scores": False,
                "uses_early_exit": False,
                "uses_compact_width_checkpoint_selection": False,
                "compact_width_validation_used_for_checkpoint": False,
                "checkpoint_selection": "full_width_validation_only",


                "parameter_count_for_runtime": "active_inference_subnetwork",
                "stored_supernet_params_reported_for_runtime": False,
            },
        },
        save_path
    )

    print("\nSaved Slimmable checkpoint:")
    print(save_path)

    return (
        global_model,
        hist,
        full_test,
        width_results,
        active_runtime_profile
    )


slimmable_model, slimmable_history, slimmable_test_metrics, slimmable_width_results, slimmable_active_runtime_profile = run_slimmable_fedavg()


In [ ]:
# ============================================================
# CELL 10: New-Client Runtime Evaluation and Charts
# ============================================================


# Imports

import os
import copy
import time
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
import matplotlib.pyplot as plt

from torch.utils.data import DataLoader, Subset


# Output paths
NEW_CLIENT_RESULTS_PATH = "CIFAR10_VGG_20NewClients_Runtime_Results_with_Slimmable.pt"
NEW_CLIENT_PER_CLIENT_CSV = "cifar10_vgg_20newclients_per_client_results_with_slimmable.csv"
NEW_CLIENT_AVG_CSV = "cifar10_vgg_20newclients_average_results_with_slimmable.csv"

RUNTIME_TABLE_CSV = "cifar10_vgg_runtime_table_params_flops_time_acc_with_slimmable.csv"

FEDAVG_PATH = "FedAvg_CIFAR10_VGG.pt"
FEDPROX_PATH = "FedProx_CIFAR10_VGG.pt"
RAS_FL_PATH = "RAS_FL_CIFAR10_VGG.pt"
FEDDROP_PATH = "FedDrop_CIFAR10_VGG.pt"
OFA_PATH = "OFA_Style_CIFAR10_VGG.pt"
SLEX_PATH = "SLEXNet_Style_CIFAR10_VGG.pt"
SLIMMABLE_PATH = "Slimmable_CIFAR10_VGG.pt"


# Evaluation settings
NUM_NEW_CLIENTS = globals().get("NUM_NEW_CLIENTS", 20)
NEW_CLIENT_NONIID_ALPHA = globals().get("NEW_CLIENT_NONIID_ALPHA", 0.05)
NEW_CLIENT_ADAPT_SAMPLES_PER_CLIENT = globals().get("NEW_CLIENT_ADAPT_SAMPLES_PER_CLIENT", 500)
NEW_CLIENT_EVAL_SAMPLES_PER_CLIENT = globals().get("NEW_CLIENT_EVAL_SAMPLES_PER_CLIENT", 500)

REMOVE_PERCENTAGES = globals().get(
    "REMOVE_PERCENTAGES",
    [5, 10, 15, 20, 25, 30, 40, 50]
)

REMOVE_PERCENTAGES = sorted(
    list(set(list(REMOVE_PERCENTAGES) + [5, 10, 15, 20, 25, 30, 40, 50]))
)

MIN_LAYER_ACTIVE_RATIO = globals().get("MIN_LAYER_ACTIVE_RATIO", 0.50)

SELECTED_REMOVE_PERCENT = 30
SELECTED_SLEX_EXIT = "exit2"

SLIMMABLE_WIDTHS = {
    0: 1.00,
    5: 0.95,
    10: 0.90,
    15: 0.85,
    20: 0.80,
    25: 0.75,
    30: 0.70,
    40: 0.60,
    50: 0.50,
}

SLEX_EXITS = ["exit1", "exit2", "exit3", "final"]

print("REMOVE_PERCENTAGES:", REMOVE_PERCENTAGES)
print("SLIMMABLE_WIDTHS:", SLIMMABLE_WIDTHS)
print("SELECTED_REMOVE_PERCENT:", SELECTED_REMOVE_PERCENT)
print("SELECTED_SLEX_EXIT:", SELECTED_SLEX_EXIT)


# Roadmap helpers
def move_generic_roadmap_to_device(roadmap, device=DEVICE):
    return {
        mode: {
            layer_name: mask.to(device)
            for layer_name, mask in layer_masks.items()
        }
        for mode, layer_masks in roadmap.items()
    }


def count_active_structures_generic(structured_mask):
    active = 0
    total = 0

    for _, mask in structured_mask.items():
        active += int(mask.sum().item())
        total += int(mask.numel())

    ratio = active / max(total, 1)

    return active, total, ratio


def load_generic_roadmap_checkpoint(path):
    checkpoint = torch.load(path, map_location=DEVICE)

    model = new_model()
    model.load_state_dict(checkpoint["model_state_dict"])
    model.to(DEVICE)
    model.eval()

    if "submodel_roadmap" in checkpoint:
        raw_roadmap = checkpoint["submodel_roadmap"]
    elif "mask_roadmap" in checkpoint:
        raw_roadmap = checkpoint["mask_roadmap"]
    else:
        raise KeyError(
            f"{path} does not contain 'submodel_roadmap' or 'mask_roadmap'."
        )

    roadmap = move_generic_roadmap_to_device(
        raw_roadmap,
        DEVICE
    )

    return model, roadmap, checkpoint


def evaluate_generic_masked_model(model, structured_mask, loader):
    temp_model = clone_model(model)

    param_masks = expand_structured_masks_to_parameter_masks(
        temp_model,
        structured_mask
    )

    apply_parameter_masks_to_model(
        temp_model,
        param_masks
    )

    temp_model.eval()

    metrics = evaluate_loss_and_metrics(
        temp_model,
        loader
    )

    active, total, ratio = count_active_structures_generic(
        structured_mask
    )

    return metrics, active, total, ratio


def make_masked_base_model(model, structured_mask):
    temp_model = clone_model(model)

    param_masks = expand_structured_masks_to_parameter_masks(
        temp_model,
        structured_mask
    )

    apply_parameter_masks_to_model(
        temp_model,
        param_masks
    )

    temp_model.eval()

    return temp_model


# SLEX helpers
def load_slex_roadmap_checkpoint(path=SLEX_PATH):
    checkpoint = torch.load(path, map_location=DEVICE)

    model = new_slex_model()
    model.load_state_dict(checkpoint["model_state_dict"])
    model.to(DEVICE)
    model.eval()

    if "submodel_roadmap" in checkpoint:
        raw_roadmap = checkpoint["submodel_roadmap"]
    elif "mask_roadmap" in checkpoint:
        raw_roadmap = checkpoint["mask_roadmap"]
    else:
        raise KeyError(
            f"{path} does not contain 'submodel_roadmap' or 'mask_roadmap'."
        )

    roadmap = move_slex_roadmap_to_device(
        raw_roadmap,
        DEVICE
    )

    return model, roadmap, checkpoint


def evaluate_slex_masked_model_on_loader(
    model,
    structured_mask,
    loader,
    exit_name
):
    temp_model = clone_slex_model(model)

    param_masks = expand_structured_masks_to_parameter_masks_slex(
        temp_model,
        structured_mask
    )

    apply_parameter_masks_to_model_slex(
        temp_model,
        param_masks
    )

    temp_model.eval()

    metrics = evaluate_slex_loss_and_metrics(
        temp_model,
        loader,
        exit_name=exit_name
    )

    active, total, ratio = count_active_structures_slex(
        structured_mask
    )

    return metrics, active, total, ratio


def make_masked_slex_model(model, structured_mask):
    temp_model = clone_slex_model(model)

    param_masks = expand_structured_masks_to_parameter_masks_slex(
        temp_model,
        structured_mask
    )

    apply_parameter_masks_to_model_slex(
        temp_model,
        param_masks
    )

    temp_model.eval()

    return temp_model


def slex_forward_to_exit(model, x, exit_name):
    """Run only the layers needed for the selected SLEX exit."""

    h1 = model.block1(x)

    if exit_name == "exit1":
        return model.exit1(h1)

    h2 = model.block2(h1)

    if exit_name == "exit2":
        return model.exit2(h2)

    h3 = model.block3(h2)

    if exit_name == "exit3":
        return model.exit3(h3)

    if exit_name == "final":
        return model.classifier(h3)

    raise ValueError(f"Unknown SLEX exit: {exit_name}")


# Slimmable helpers
def set_slimmable_width(model, width_mult):
    width_mult = float(width_mult)

    if hasattr(model, "set_active_width"):
        model.set_active_width(width_mult)
        return

    if hasattr(model, "set_width_mult"):
        model.set_width_mult(width_mult)
        return

    changed = False

    for module in model.modules():
        if hasattr(module, "set_active_width"):
            module.set_active_width(width_mult)
            changed = True

        elif hasattr(module, "set_width_mult"):
            module.set_width_mult(width_mult)
            changed = True

        elif hasattr(module, "active_width"):
            module.active_width = width_mult
            changed = True

        elif hasattr(module, "width_mult"):
            module.width_mult = width_mult
            changed = True

    if not changed:
        raise AttributeError(
            "Could not set Slimmable width. "
            "Please check that Cell 9 defines new_slimmable_model() and set_active_width()."
        )


def load_slimmable_checkpoint(path=SLIMMABLE_PATH):
    if not os.path.exists(path):
        raise FileNotFoundError(
            f"Missing Slimmable checkpoint: {path}. "
            "Run the updated Cell 9 first."
        )

    checkpoint = torch.load(path, map_location=DEVICE)

    model = new_slimmable_model()

    if isinstance(checkpoint, dict) and "model_state_dict" in checkpoint:
        model.load_state_dict(checkpoint["model_state_dict"])

    elif isinstance(checkpoint, dict) and "state_dict" in checkpoint:
        model.load_state_dict(checkpoint["state_dict"])

    else:
        model.load_state_dict(checkpoint)

    model.to(DEVICE)
    model.eval()

    return model, checkpoint


def make_slimmable_model_at_width(model, width_mult):
    temp_model = copy.deepcopy(model)
    temp_model.to(DEVICE)
    temp_model.eval()

    set_slimmable_width(temp_model, width_mult)

    return temp_model


def count_slimmable_active_params_cifar10_vgg(width_mult):
    """Count active Slimmable parameters for one width."""

    width_mult = float(width_mult)

    c1 = max(1, int(round(64 * width_mult)))
    c2 = max(1, int(round(64 * width_mult)))
    c3 = max(1, int(round(128 * width_mult)))
    c4 = max(1, int(round(128 * width_mult)))
    c5 = max(1, int(round(256 * width_mult)))
    c6 = max(1, int(round(256 * width_mult)))
    h1 = max(1, int(round(512 * width_mult)))

    params = 0

    params += c1 * 3 * 3 * 3
    params += 2 * c1

    params += c2 * c1 * 3 * 3
    params += 2 * c2

    params += c3 * c2 * 3 * 3
    params += 2 * c3

    params += c4 * c3 * 3 * 3
    params += 2 * c4

    params += c5 * c4 * 3 * 3
    params += 2 * c5

    params += c6 * c5 * 3 * 3
    params += 2 * c6

    params += h1 * (c6 * 4 * 4)

    params += 2 * h1

    params += NUM_CLASSES * h1
    params += NUM_CLASSES

    return int(params)


def count_slimmable_active_flops_cifar10_vgg(width_mult):
    """Estimate active Slimmable FLOPs for one width."""

    width_mult = float(width_mult)

    c1 = max(1, int(round(64 * width_mult)))
    c2 = max(1, int(round(64 * width_mult)))
    c3 = max(1, int(round(128 * width_mult)))
    c4 = max(1, int(round(128 * width_mult)))
    c5 = max(1, int(round(256 * width_mult)))
    c6 = max(1, int(round(256 * width_mult)))
    h1 = max(1, int(round(512 * width_mult)))

    flops = 0

    flops += 32 * 32 * c1 * 3 * 3 * 3
    flops += 32 * 32 * c2 * c1 * 3 * 3

    flops += 16 * 16 * c3 * c2 * 3 * 3
    flops += 16 * 16 * c4 * c3 * 3 * 3

    flops += 8 * 8 * c5 * c4 * 3 * 3
    flops += 8 * 8 * c6 * c5 * 3 * 3

    flops += (c6 * 4 * 4) * h1

    flops += h1 * NUM_CLASSES

    return int(flops)


def get_slimmable_runtime_from_checkpoint(
    slimmable_ckpt,
    remove_p,
    width_mult
):
    """Read active runtime values from the checkpoint, with a fallback."""

    width_mult = float(width_mult)
    remove_key = f"remove_{int(remove_p)}"
    width_key_1 = str(float(round(width_mult, 2)))
    width_key_2 = str(round(width_mult, 2))

    if isinstance(slimmable_ckpt, dict):
        if (
            "active_params_by_remove_percent" in slimmable_ckpt
            and "active_flops_by_remove_percent" in slimmable_ckpt
            and remove_key in slimmable_ckpt["active_params_by_remove_percent"]
            and remove_key in slimmable_ckpt["active_flops_by_remove_percent"]
        ):
            params = int(
                slimmable_ckpt["active_params_by_remove_percent"][remove_key]
            )
            flops = int(
                slimmable_ckpt["active_flops_by_remove_percent"][remove_key]
            )
            return params, flops

        if (
            "active_params_by_width" in slimmable_ckpt
            and "active_flops_by_width" in slimmable_ckpt
        ):
            if width_key_1 in slimmable_ckpt["active_params_by_width"]:
                params = int(
                    slimmable_ckpt["active_params_by_width"][width_key_1]
                )
                flops = int(
                    slimmable_ckpt["active_flops_by_width"][width_key_1]
                )
                return params, flops

            if width_key_2 in slimmable_ckpt["active_params_by_width"]:
                params = int(
                    slimmable_ckpt["active_params_by_width"][width_key_2]
                )
                flops = int(
                    slimmable_ckpt["active_flops_by_width"][width_key_2]
                )
                return params, flops

    params = count_slimmable_active_params_cifar10_vgg(width_mult)
    flops = count_slimmable_active_flops_cifar10_vgg(width_mult)

    return params, flops


def evaluate_slimmable_model_on_loader(
    model,
    checkpoint,
    remove_p,
    width_mult,
    loader
):
    temp_model = copy.deepcopy(model)
    temp_model.to(DEVICE)
    temp_model.eval()

    width_mult = float(width_mult)
    set_slimmable_width(temp_model, width_mult)

    metrics = evaluate_loss_and_metrics(
        temp_model,
        loader
    )

    active_params, active_flops = get_slimmable_runtime_from_checkpoint(
        checkpoint,
        remove_p,
        width_mult
    )

    if isinstance(checkpoint, dict) and "stored_supernet_params" in checkpoint:
        total_params = int(checkpoint["stored_supernet_params"])
    else:
        total_params = count_slimmable_active_params_cifar10_vgg(1.00)

    active = active_params
    total = total_params
    ratio = width_mult

    return metrics, active, total, ratio, active_params, active_flops


# New-client data
def sample_class_counts_for_new_client(
    num_samples,
    alpha,
    num_classes,
    seed
):
    rng = np.random.default_rng(seed)

    probs = rng.dirichlet(
        alpha * np.ones(num_classes)
    )

    counts = rng.multinomial(
        num_samples,
        probs
    )

    return counts


def create_loader_from_class_counts(
    dataset,
    class_counts,
    seed,
    batch_size=BATCH_SIZE,
    shuffle=False
):
    rng = np.random.default_rng(seed)

    targets = get_targets(dataset)

    chosen_indices = []

    for c, count in enumerate(class_counts):
        class_indices = np.where(targets == c)[0]

        if len(class_indices) == 0 or count == 0:
            continue

        replace = count > len(class_indices)

        sampled = rng.choice(
            class_indices,
            size=count,
            replace=replace
        )

        chosen_indices.extend(sampled.tolist())

    rng.shuffle(chosen_indices)

    loader = DataLoader(
        Subset(dataset, chosen_indices),
        batch_size=batch_size,
        shuffle=shuffle,
        num_workers=2,
        pin_memory=torch.cuda.is_available()
    )

    return loader, chosen_indices


def build_20_new_clients():
    adapt_loaders = []
    eval_loaders = []
    adapt_indices_list = []
    eval_indices_list = []
    distributions = []

    for cid in range(NUM_NEW_CLIENTS):
        adapt_counts = sample_class_counts_for_new_client(
            num_samples=NEW_CLIENT_ADAPT_SAMPLES_PER_CLIENT,
            alpha=NEW_CLIENT_NONIID_ALPHA,
            num_classes=NUM_CLASSES,
            seed=SEED + 9000 + cid
        )

        eval_counts = sample_class_counts_for_new_client(
            num_samples=NEW_CLIENT_EVAL_SAMPLES_PER_CLIENT,
            alpha=NEW_CLIENT_NONIID_ALPHA,
            num_classes=NUM_CLASSES,
            seed=SEED + 10000 + cid
        )

        adapt_loader, adapt_indices = create_loader_from_class_counts(
            dataset=test_dataset,
            class_counts=adapt_counts,
            seed=SEED + 11000 + cid,
            shuffle=False
        )

        eval_loader, eval_indices = create_loader_from_class_counts(
            dataset=test_dataset,
            class_counts=eval_counts,
            seed=SEED + 12000 + cid,
            shuffle=False
        )

        adapt_loaders.append(adapt_loader)
        eval_loaders.append(eval_loader)
        adapt_indices_list.append(adapt_indices)
        eval_indices_list.append(eval_indices)

        distributions.append({
            "client_id": cid,
            "adapt_counts": adapt_counts,
            "eval_counts": eval_counts,
        })

    return (
        adapt_loaders,
        eval_loaders,
        adapt_indices_list,
        eval_indices_list,
        distributions
    )


# RAS-FL personalization
def mix_ras_fl_importance(
    global_importance,
    local_importance,
    num_local_samples,
    gamma=IMPORTANCE_GAMMA
):
    lambda_k = 1.0 / (1.0 + gamma * num_local_samples)

    mixed = {}

    for name in global_importance:
        mixed[name] = (
            lambda_k * global_importance[name]
            + (1.0 - lambda_k) * local_importance[name]
        )

    return mixed, lambda_k


# Load trained models
def load_all_trained_models_for_cell10():
    required_paths = {
        "FedAvg": FEDAVG_PATH,
        "FedProx": FEDPROX_PATH,
        "RAS-FL": RAS_FL_PATH,
        "FedDrop": FEDDROP_PATH,
        "OFA": OFA_PATH,
        "SLEX": SLEX_PATH,
        "Slimmable": SLIMMABLE_PATH,
    }

    for name, path in required_paths.items():
        if not os.path.exists(path):
            raise FileNotFoundError(
                f"Missing checkpoint for {name}: {path}. "
                "Run the corresponding training cell first."
            )

    fedavg_model, fedavg_ckpt = load_full_model_checkpoint(FEDAVG_PATH)
    fedprox_model, fedprox_ckpt = load_full_model_checkpoint(FEDPROX_PATH)

    ras_fl_model, ras_fl_global_importance, ras_fl_global_roadmap, ras_fl_ckpt = load_ras_fl_checkpoint(
        RAS_FL_PATH
    )

    feddrop_model, feddrop_roadmap, feddrop_ckpt = load_generic_roadmap_checkpoint(
        FEDDROP_PATH
    )

    ofa_model, ofa_roadmap, ofa_ckpt = load_generic_roadmap_checkpoint(
        OFA_PATH
    )

    slex_model, slex_roadmap, slex_ckpt = load_slex_roadmap_checkpoint(
        SLEX_PATH
    )

    slimmable_model, slimmable_ckpt = load_slimmable_checkpoint(
        SLIMMABLE_PATH
    )

    loaded = {
        "fedavg_model": fedavg_model,
        "fedavg_ckpt": fedavg_ckpt,

        "fedprox_model": fedprox_model,
        "fedprox_ckpt": fedprox_ckpt,

        "ras_fl_model": ras_fl_model,
        "ras_fl_global_importance": ras_fl_global_importance,
        "ras_fl_global_roadmap": ras_fl_global_roadmap,
        "ras_fl_ckpt": ras_fl_ckpt,

        "feddrop_model": feddrop_model,
        "feddrop_roadmap": feddrop_roadmap,
        "feddrop_ckpt": feddrop_ckpt,

        "ofa_model": ofa_model,
        "ofa_roadmap": ofa_roadmap,
        "ofa_ckpt": ofa_ckpt,

        "slex_model": slex_model,
        "slex_roadmap": slex_roadmap,
        "slex_ckpt": slex_ckpt,

        "slimmable_model": slimmable_model,
        "slimmable_ckpt": slimmable_ckpt,
    }

    print("Loaded all trained models/checkpoints for Cell 10.")

    return loaded


loaded_models = load_all_trained_models_for_cell10()


# Evaluate one new client
def evaluate_all_methods_on_new_client(
    client_id,
    adapt_loader,
    eval_loader,
    loaded
):
    rows = []

    fedavg_model = loaded["fedavg_model"]
    fedprox_model = loaded["fedprox_model"]

    ras_fl_model = loaded["ras_fl_model"]
    ras_fl_global_importance = loaded["ras_fl_global_importance"]
    ras_fl_global_roadmap = loaded["ras_fl_global_roadmap"]

    feddrop_model = loaded["feddrop_model"]
    feddrop_roadmap = loaded["feddrop_roadmap"]

    ofa_model = loaded["ofa_model"]
    ofa_roadmap = loaded["ofa_roadmap"]

    slex_model = loaded["slex_model"]
    slex_roadmap = loaded["slex_roadmap"]

    slimmable_model = loaded["slimmable_model"]
    slimmable_ckpt = loaded["slimmable_ckpt"]


    for family, method_name, model in [
        ("FedAvg", "FedAvg_full", fedavg_model),
        ("FedProx", "FedProx_full", fedprox_model),
    ]:
        metrics = evaluate_loss_and_metrics(
            model,
            eval_loader
        )

        rows.append({
            "client_id": client_id,
            "family": family,
            "method": method_name,
            "roadmap_type": "full_model",
            "exit": "none",
            "remove_low_percent": 0,
            "accuracy": metrics["acc"],
            "loss": metrics["loss"],
            "macro_f1": metrics["macro_f1"],
            "balanced_acc": metrics["balanced_acc"],
            "active_structures": np.nan,
            "total_structures": np.nan,
            "structure_active_ratio": 1.00,
            "active_params": np.nan,
            "active_flops": np.nan,
            "personalization_lambda": np.nan,
        })


    for mode_name, structured_mask in ras_fl_global_roadmap.items():
        if mode_name == "full":
            remove_p = 0
        elif mode_name.startswith("remove_low_"):
            remove_p = int(mode_name.replace("remove_low_", ""))
        else:
            continue

        metrics, active, total, ratio = evaluate_generic_masked_model(
            model=ras_fl_model,
            structured_mask=structured_mask,
            loader=eval_loader
        )

        rows.append({
            "client_id": client_id,
            "family": "RAS-FL",
            "method": f"RAS-FL_global_{mode_name}",
            "roadmap_type": "ras_fl_global_importance",
            "exit": "none",
            "remove_low_percent": remove_p,
            "accuracy": metrics["acc"],
            "loss": metrics["loss"],
            "macro_f1": metrics["macro_f1"],
            "balanced_acc": metrics["balanced_acc"],
            "active_structures": active,
            "total_structures": total,
            "structure_active_ratio": ratio,
            "active_params": np.nan,
            "active_flops": np.nan,
            "personalization_lambda": np.nan,
        })


    local_importance = estimate_local_structured_importance(
        clone_model(ras_fl_model),
        adapt_loader,
        max_batches=IMPORTANCE_MAX_BATCHES
    )

    ras_fl_personalized_importance, lambda_k = mix_ras_fl_importance(
        global_importance=ras_fl_global_importance,
        local_importance=local_importance,
        num_local_samples=NEW_CLIENT_ADAPT_SAMPLES_PER_CLIENT,
        gamma=IMPORTANCE_GAMMA
    )

    ras_fl_personalized_roadmap, ras_fl_personalized_metadata = build_boundary_aware_importance_roadmap(
        model=ras_fl_model,
        importance=ras_fl_personalized_importance,
        remove_percentages=REMOVE_PERCENTAGES
    )

    for mode_name, structured_mask in ras_fl_personalized_roadmap.items():
        if mode_name == "full":
            remove_p = 0
        elif mode_name.startswith("remove_low_"):
            remove_p = int(mode_name.replace("remove_low_", ""))
        else:
            continue

        metrics, active, total, ratio = evaluate_generic_masked_model(
            model=ras_fl_model,
            structured_mask=structured_mask,
            loader=eval_loader
        )

        rows.append({
            "client_id": client_id,
            "family": "RAS-FL",
            "method": f"RAS-FL_personalized_{mode_name}",
            "roadmap_type": "ras_fl_personalized_importance",
            "exit": "none",
            "remove_low_percent": remove_p,
            "accuracy": metrics["acc"],
            "loss": metrics["loss"],
            "macro_f1": metrics["macro_f1"],
            "balanced_acc": metrics["balanced_acc"],
            "active_structures": active,
            "total_structures": total,
            "structure_active_ratio": ratio,
            "active_params": np.nan,
            "active_flops": np.nan,
            "personalization_lambda": lambda_k,
        })


    for mode_name, structured_mask in feddrop_roadmap.items():
        if mode_name == "full":
            remove_p = 0
        elif mode_name.startswith("feddrop_remove_"):
            remove_p = int(mode_name.replace("feddrop_remove_", ""))
        else:
            continue

        metrics, active, total, ratio = evaluate_generic_masked_model(
            model=feddrop_model,
            structured_mask=structured_mask,
            loader=eval_loader
        )

        rows.append({
            "client_id": client_id,
            "family": "FedDrop",
            "method": f"FedDrop_{mode_name}",
            "roadmap_type": "random_boundary",
            "exit": "none",
            "remove_low_percent": remove_p,
            "accuracy": metrics["acc"],
            "loss": metrics["loss"],
            "macro_f1": metrics["macro_f1"],
            "balanced_acc": metrics["balanced_acc"],
            "active_structures": active,
            "total_structures": total,
            "structure_active_ratio": ratio,
            "active_params": np.nan,
            "active_flops": np.nan,
            "personalization_lambda": np.nan,
        })


    for mode_name, structured_mask in ofa_roadmap.items():
        if mode_name == "full":
            remove_p = 0
        elif mode_name.startswith("ofa_remove_"):
            remove_p = int(mode_name.replace("ofa_remove_", ""))
        else:
            continue

        metrics, active, total, ratio = evaluate_generic_masked_model(
            model=ofa_model,
            structured_mask=structured_mask,
            loader=eval_loader
        )

        rows.append({
            "client_id": client_id,
            "family": "OFA",
            "method": f"OFA_{mode_name}",
            "roadmap_type": "predefined_nested_width",
            "exit": "none",
            "remove_low_percent": remove_p,
            "accuracy": metrics["acc"],
            "loss": metrics["loss"],
            "macro_f1": metrics["macro_f1"],
            "balanced_acc": metrics["balanced_acc"],
            "active_structures": active,
            "total_structures": total,
            "structure_active_ratio": ratio,
            "active_params": np.nan,
            "active_flops": np.nan,
            "personalization_lambda": np.nan,
        })


    for mode_name, structured_mask in slex_roadmap.items():
        if mode_name == "full":
            remove_p = 0
        elif mode_name.startswith("slex_remove_"):
            remove_p = int(mode_name.replace("slex_remove_", ""))
        else:
            continue

        for exit_name in SLEX_EXITS:
            metrics, active, total, ratio = evaluate_slex_masked_model_on_loader(
                model=slex_model,
                structured_mask=structured_mask,
                loader=eval_loader,
                exit_name=exit_name
            )

            rows.append({
                "client_id": client_id,
                "family": "SLEX",
                "method": f"SLEX_{mode_name}_{exit_name}",
                "roadmap_type": "pure_slex_width_early_exit",
                "exit": exit_name,
                "remove_low_percent": remove_p,
                "accuracy": metrics["acc"],
                "loss": metrics["loss"],
                "macro_f1": metrics["macro_f1"],
                "balanced_acc": metrics["balanced_acc"],
                "active_structures": active,
                "total_structures": total,
                "structure_active_ratio": ratio,
                "active_params": np.nan,
                "active_flops": np.nan,
                "personalization_lambda": np.nan,
            })


    for remove_p, width_mult in SLIMMABLE_WIDTHS.items():
        width_mult = max(float(width_mult), MIN_LAYER_ACTIVE_RATIO)

        (
            metrics,
            active,
            total,
            ratio,
            active_params,
            active_flops,
        ) = evaluate_slimmable_model_on_loader(
            model=slimmable_model,
            checkpoint=slimmable_ckpt,
            remove_p=remove_p,
            width_mult=width_mult,
            loader=eval_loader
        )

        rows.append({
            "client_id": client_id,
            "family": "Slimmable",
            "method": f"Slimmable_width_{width_mult:.2f}",
            "roadmap_type": "slimmable_width",
            "exit": "none",
            "remove_low_percent": remove_p,
            "accuracy": metrics["acc"],
            "loss": metrics["loss"],
            "macro_f1": metrics["macro_f1"],
            "balanced_acc": metrics["balanced_acc"],
            "active_structures": active,
            "total_structures": total,
            "structure_active_ratio": ratio,
            "active_params": active_params,
            "active_flops": active_flops,
            "personalization_lambda": np.nan,
        })

    return rows


# Evaluate all new clients
(
    new_client_adapt_loaders,
    new_client_eval_loaders,
    new_client_adapt_indices_list,
    new_client_eval_indices_list,
    new_client_distributions
) = build_20_new_clients()

all_rows = []

for cid in range(NUM_NEW_CLIENTS):
    print(f"\nEvaluating new client {cid + 1}/{NUM_NEW_CLIENTS}...")

    client_rows = evaluate_all_methods_on_new_client(
        client_id=cid,
        adapt_loader=new_client_adapt_loaders[cid],
        eval_loader=new_client_eval_loaders[cid],
        loaded=loaded_models
    )

    all_rows.extend(client_rows)

new_client_results_df = pd.DataFrame(all_rows)

if "exit" not in new_client_results_df.columns:
    new_client_results_df["exit"] = "none"

new_client_results_df["exit"] = new_client_results_df["exit"].fillna("none")


# Aggregate results
group_cols = [
    "family",
    "roadmap_type",
    "exit",
    "remove_low_percent"
]

avg_results_df = (
    new_client_results_df
    .groupby(group_cols, dropna=False)
    .agg(
        accuracy_mean=("accuracy", "mean"),
        accuracy_std=("accuracy", "std"),
        loss_mean=("loss", "mean"),
        loss_std=("loss", "std"),
        macro_f1_mean=("macro_f1", "mean"),
        macro_f1_std=("macro_f1", "std"),
        balanced_acc_mean=("balanced_acc", "mean"),
        balanced_acc_std=("balanced_acc", "std"),
        structure_active_ratio_mean=("structure_active_ratio", "mean"),
        structure_active_ratio_std=("structure_active_ratio", "std"),
        active_structures_mean=("active_structures", "mean"),
        total_structures_mean=("total_structures", "mean"),
        active_params_mean=("active_params", "mean"),
        active_params_std=("active_params", "std"),
        active_flops_mean=("active_flops", "mean"),
        active_flops_std=("active_flops", "std"),
        personalization_lambda_mean=("personalization_lambda", "mean"),
    )
    .reset_index()
)

print("\nPer-client results:")
display(new_client_results_df)

print("\nAverage results across 20 new clients:")
display(avg_results_df)


# Save results
new_client_results_df.to_csv(
    NEW_CLIENT_PER_CLIENT_CSV,
    index=False
)

avg_results_df.to_csv(
    NEW_CLIENT_AVG_CSV,
    index=False
)

torch.save(
    {
        "per_client_results": new_client_results_df,
        "average_results": avg_results_df,
        "new_client_distributions": new_client_distributions,
        "config": {
            "dataset": "CIFAR-10",
            "architecture": "VGG-style",
            "num_classes": NUM_CLASSES,
            "num_new_clients": NUM_NEW_CLIENTS,
            "new_client_noniid_alpha": NEW_CLIENT_NONIID_ALPHA,
            "adapt_samples_per_client": NEW_CLIENT_ADAPT_SAMPLES_PER_CLIENT,
            "eval_samples_per_client": NEW_CLIENT_EVAL_SAMPLES_PER_CLIENT,
            "remove_percentages": REMOVE_PERCENTAGES,
            "min_layer_active_ratio": MIN_LAYER_ACTIVE_RATIO,
            "slimmable_widths": SLIMMABLE_WIDTHS,
            "slimmable_runtime_accounting": "active_inference_subnetwork",
            "slimmable_stored_supernet_reported_for_runtime": False,
            "methods": [
                "FedAvg",
                "FedProx",
                "RAS-FL global roadmap",
                "RAS-FL personalized roadmap",
                "Federated Dropout-style",
                "OFA-style",
                "SLEX-style",
                "Slimmable Neural Networks",
            ],
            "slex_exits": SLEX_EXITS,
        },
    },
    NEW_CLIENT_RESULTS_PATH
)

print("\nSaved:")
print(NEW_CLIENT_RESULTS_PATH)
print(NEW_CLIENT_PER_CLIENT_CSV)
print(NEW_CLIENT_AVG_CSV)


# Plot labels
def get_plot_label(row):
    family = str(row.get("family", ""))
    roadmap_type = str(row.get("roadmap_type", ""))
    exit_name = str(row.get("exit", "none"))

    if family == "FedAvg":
        return "FedAvg"

    if family == "FedProx":
        return "FedProx"

    if roadmap_type == "ras_fl_global_importance":
        return "RAS-FL global"

    if roadmap_type == "ras_fl_personalized_importance":
        return "RAS-FL personalized"

    if roadmap_type == "random_boundary":
        return "Federated Dropout-style"

    if roadmap_type == "predefined_nested_width":
        return "OFA-style"

    if roadmap_type == "pure_slex_width_early_exit":
        return f"SLEX-style {exit_name}"

    if roadmap_type == "slimmable_width":
        return "Slimmable"

    return f"{family}_{roadmap_type}_{exit_name}"


avg_results_df["plot_label"] = avg_results_df.apply(get_plot_label, axis=1)
new_client_results_df["plot_label"] = new_client_results_df.apply(get_plot_label, axis=1)


# Accuracy charts
plt.figure(figsize=(13.5, 7))

for label, group in avg_results_df.groupby("plot_label"):
    group = group.sort_values("remove_low_percent")

    plt.errorbar(
        group["remove_low_percent"],
        group["accuracy_mean"] * 100.0,
        yerr=group["accuracy_std"] * 100.0,
        marker="o",
        capsize=3,
        linewidth=2,
        label=label
    )

plt.xlabel("Removed Structures (%)", fontweight="bold")
plt.ylabel("Average Accuracy Across 20 New Clients (%)", fontweight="bold")
plt.title("CIFAR-10 + VGG: 20 New-Client Runtime Evaluation", fontweight="bold")
plt.grid(alpha=0.3)
plt.legend(ncol=2)
plt.tight_layout()
plt.savefig(
    "cifar10_vgg_20newclients_accuracy_vs_removed_structures_with_slimmable.png",
    dpi=400,
    bbox_inches="tight"
)
plt.savefig(
    "cifar10_vgg_20newclients_accuracy_vs_removed_structures_with_slimmable.pdf",
    bbox_inches="tight"
)
plt.show()


plt.figure(figsize=(13.5, 7))

plot_df = avg_results_df.dropna(
    subset=["structure_active_ratio_mean"]
).copy()

for label, group in plot_df.groupby("plot_label"):
    group = group.sort_values("structure_active_ratio_mean")

    plt.errorbar(
        group["structure_active_ratio_mean"],
        group["accuracy_mean"] * 100.0,
        yerr=group["accuracy_std"] * 100.0,
        marker="o",
        capsize=3,
        linewidth=2,
        label=label
    )

plt.xlabel("Active Structure Ratio", fontweight="bold")
plt.ylabel("Average Accuracy Across 20 New Clients (%)", fontweight="bold")
plt.title("CIFAR-10 + VGG: Accuracy vs Active Structure Ratio", fontweight="bold")
plt.grid(alpha=0.3)
plt.legend(ncol=2)
plt.tight_layout()
plt.savefig(
    "cifar10_vgg_20newclients_accuracy_vs_active_ratio_with_slimmable.png",
    dpi=400,
    bbox_inches="tight"
)
plt.savefig(
    "cifar10_vgg_20newclients_accuracy_vs_active_ratio_with_slimmable.pdf",
    bbox_inches="tight"
)
plt.show()


full_subset = avg_results_df[
    avg_results_df["remove_low_percent"].fillna(0).astype(int) == 0
].copy()

full_subset = full_subset[
    ~(
        (full_subset["roadmap_type"] == "pure_slex_width_early_exit")
        & (full_subset["exit"] != "final")
    )
].copy()

plt.figure(figsize=(12.5, 6))

x = np.arange(len(full_subset))

plt.bar(
    x,
    full_subset["accuracy_mean"] * 100.0,
    yerr=full_subset["accuracy_std"] * 100.0,
    capsize=4,
    edgecolor="black",
    linewidth=1.0
)

plt.xticks(
    x,
    full_subset["plot_label"],
    rotation=30,
    ha="right",
    fontweight="bold"
)

plt.ylabel("Average Accuracy Across 20 New Clients (%)", fontweight="bold")
plt.xlabel("Method", fontweight="bold")
plt.title("CIFAR-10 + VGG: Full / Unshrunken Accuracy", fontweight="bold")
plt.grid(axis="y", alpha=0.3)
plt.tight_layout()
plt.savefig(
    "cifar10_vgg_20newclients_full_accuracy_bar_with_slimmable.png",
    dpi=400,
    bbox_inches="tight"
)
plt.savefig(
    "cifar10_vgg_20newclients_full_accuracy_bar_with_slimmable.pdf",
    bbox_inches="tight"
)
plt.show()


box_specs = [
    ("FedAvg", "full_model", "none", 0, "FedAvg"),
    ("FedProx", "full_model", "none", 0, "FedProx"),
    ("FedDrop", "random_boundary", "none", SELECTED_REMOVE_PERCENT, "Federated Dropout-style"),
    ("OFA", "predefined_nested_width", "none", SELECTED_REMOVE_PERCENT, "OFA-style"),
    ("SLEX", "pure_slex_width_early_exit", SELECTED_SLEX_EXIT, SELECTED_REMOVE_PERCENT, f"SLEX-style {SELECTED_SLEX_EXIT}"),
    ("Slimmable", "slimmable_width", "none", SELECTED_REMOVE_PERCENT, "Slimmable"),
    ("RAS-FL", "ras_fl_global_importance", "none", SELECTED_REMOVE_PERCENT, "RAS-FL global"),
    ("RAS-FL", "ras_fl_personalized_importance", "none", SELECTED_REMOVE_PERCENT, "RAS-FL personalized"),
]

box_data = []
box_labels = []

for family, roadmap_type, exit_name, remove_p, label in box_specs:
    temp = new_client_results_df[
        (new_client_results_df["family"] == family)
        & (new_client_results_df["roadmap_type"] == roadmap_type)
        & (new_client_results_df["exit"] == exit_name)
        & (new_client_results_df["remove_low_percent"] == remove_p)
    ]

    if len(temp) > 0:
        box_data.append(temp["accuracy"].values * 100.0)
        box_labels.append(label)
    else:
        print(f"Warning: no boxplot data for {label}")

plt.figure(figsize=(13.5, 6.5))

plt.boxplot(
    box_data,
    labels=box_labels,
    showmeans=True,
    patch_artist=True
)

plt.xticks(rotation=30, ha="right", fontweight="bold")
plt.ylabel("Per-Client Accuracy (%)", fontweight="bold")
plt.xlabel("Method", fontweight="bold")
plt.title(
    f"CIFAR-10 + VGG: Per-Client Accuracy Distribution "
    f"(remove={SELECTED_REMOVE_PERCENT}%, SLEX={SELECTED_SLEX_EXIT})",
    fontweight="bold"
)
plt.grid(axis="y", alpha=0.3)
plt.tight_layout()
plt.savefig(
    "cifar10_vgg_20newclients_boxplot_selected_runtime_with_slimmable.png",
    dpi=400,
    bbox_inches="tight"
)
plt.savefig(
    "cifar10_vgg_20newclients_boxplot_selected_runtime_with_slimmable.pdf",
    bbox_inches="tight"
)
plt.show()


# Runtime profiling
def profile_effective_params_flops(
    model,
    input_size=(1, 3, 32, 32),
    slex_exit=None
):
    """Estimate active Conv2d and Linear parameters and FLOPs."""

    model = model.to(DEVICE)
    model.eval()

    layer_stats = []
    hooks = []

    def regular_conv_hook(module, inp, out):
        x = inp[0].detach()
        y = out.detach()

        active_in = int(
            (x.abs().sum(dim=(0, 2, 3)) > 1e-8).sum().item()
        )
        active_out = int(
            (y.abs().sum(dim=(0, 2, 3)) > 1e-8).sum().item()
        )

        kh, kw = module.kernel_size
        out_h, out_w = y.shape[2], y.shape[3]

        params = active_out * active_in * kh * kw

        if module.bias is not None:
            params += active_out

        flops = out_h * out_w * active_out * active_in * kh * kw

        layer_stats.append({
            "type": "conv",
            "params": params,
            "flops": flops,
        })

    def regular_linear_hook(module, inp, out):
        x = inp[0].detach()
        y = out.detach()

        if x.dim() > 2:
            x = torch.flatten(x, 1)

        active_in = int(
            (x.abs().sum(dim=0) > 1e-8).sum().item()
        )
        active_out = int(
            (y.abs().sum(dim=0) > 1e-8).sum().item()
        )

        params = active_out * active_in

        if module.bias is not None:
            params += active_out

        flops = active_in * active_out

        layer_stats.append({
            "type": "linear",
            "params": params,
            "flops": flops,
        })

    for module in model.modules():
        if isinstance(module, nn.Conv2d):
            hooks.append(
                module.register_forward_hook(regular_conv_hook)
            )

        elif isinstance(module, nn.Linear):
            hooks.append(
                module.register_forward_hook(regular_linear_hook)
            )

    with torch.no_grad():
        dummy = torch.ones(input_size, device=DEVICE)

        if slex_exit is None:
            _ = model(dummy)
        else:
            _ = slex_forward_to_exit(model, dummy, slex_exit)

    for h in hooks:
        h.remove()

    total_params = int(
        sum(x["params"] for x in layer_stats)
    )
    total_flops = int(
        sum(x["flops"] for x in layer_stats)
    )

    return total_params, total_flops


def measure_base_inference_time(model, eval_loaders):
    model.eval()
    total_time = 0.0

    with torch.no_grad():
        for loader in eval_loaders:
            if torch.cuda.is_available():
                torch.cuda.synchronize()

            start = time.perf_counter()

            for x, _ in loader:
                x = x.to(DEVICE)
                _ = model(x)

            if torch.cuda.is_available():
                torch.cuda.synchronize()

            total_time += time.perf_counter() - start

    return total_time / max(len(eval_loaders), 1)


def measure_slex_inference_time(model, eval_loaders, exit_name):
    model.eval()
    total_time = 0.0

    with torch.no_grad():
        for loader in eval_loaders:
            if torch.cuda.is_available():
                torch.cuda.synchronize()

            start = time.perf_counter()

            for x, _ in loader:
                x = x.to(DEVICE)
                _ = slex_forward_to_exit(model, x, exit_name)

            if torch.cuda.is_available():
                torch.cuda.synchronize()

            total_time += time.perf_counter() - start

    return total_time / max(len(eval_loaders), 1)


def fmt_params(n):
    n = float(n)

    if n >= 1e6:
        return f"{n / 1e6:.2f}M"

    if n >= 1e3:
        return f"{n / 1e3:.2f}K"

    return str(int(n))


def fmt_flops(n):
    n = float(n)

    if n >= 1e9:
        return f"{n / 1e9:.2f}G"

    if n >= 1e6:
        return f"{n / 1e6:.2f}M"

    if n >= 1e3:
        return f"{n / 1e3:.2f}K"

    return str(int(n))


def get_acc_from_avg(
    family,
    roadmap_type,
    remove_p,
    exit_name="none"
):
    row = avg_results_df[
        (avg_results_df["family"] == family)
        & (avg_results_df["roadmap_type"] == roadmap_type)
        & (avg_results_df["remove_low_percent"] == remove_p)
        & (avg_results_df["exit"] == exit_name)
    ]

    if len(row) == 0:
        return np.nan

    return float(row.iloc[0]["accuracy_mean"])


# Build the runtime table
runtime_rows = []


fedavg_params, fedavg_flops = profile_effective_params_flops(
    loaded_models["fedavg_model"]
)

fedavg_time = measure_base_inference_time(
    loaded_models["fedavg_model"],
    new_client_eval_loaders
)

runtime_rows.append({
    "Method": "FedAvg",
    "Activated": "full",
    "Params": fedavg_params,
    "FLOPs": fedavg_flops,
    "Time(s)": fedavg_time,
    "Acc.": get_acc_from_avg("FedAvg", "full_model", 0, "none"),
})


fedprox_params, fedprox_flops = profile_effective_params_flops(
    loaded_models["fedprox_model"]
)

fedprox_time = measure_base_inference_time(
    loaded_models["fedprox_model"],
    new_client_eval_loaders
)

runtime_rows.append({
    "Method": "FedProx",
    "Activated": "full",
    "Params": fedprox_params,
    "FLOPs": fedprox_flops,
    "Time(s)": fedprox_time,
    "Acc.": get_acc_from_avg("FedProx", "full_model", 0, "none"),
})


for mode_name, structured_mask in loaded_models["ras_fl_global_roadmap"].items():
    if mode_name == "full":
        remove_p = 0
    elif mode_name.startswith("remove_low_"):
        remove_p = int(mode_name.replace("remove_low_", ""))
    else:
        continue

    temp_model = make_masked_base_model(
        loaded_models["ras_fl_model"],
        structured_mask
    )

    params, flops = profile_effective_params_flops(
        temp_model
    )

    avg_time = measure_base_inference_time(
        temp_model,
        new_client_eval_loaders
    )

    runtime_rows.append({
        "Method": "RAS-FL global",
        "Activated": f"remove_{remove_p}",
        "Params": params,
        "FLOPs": flops,
        "Time(s)": avg_time,
        "Acc.": get_acc_from_avg(
            "RAS-FL",
            "ras_fl_global_importance",
            remove_p,
            "none"
        ),
    })


for mode_name, structured_mask in loaded_models["ras_fl_global_roadmap"].items():
    if mode_name == "full":
        remove_p = 0
    elif mode_name.startswith("remove_low_"):
        remove_p = int(mode_name.replace("remove_low_", ""))
    else:
        continue

    temp_model = make_masked_base_model(
        loaded_models["ras_fl_model"],
        structured_mask
    )

    params, flops = profile_effective_params_flops(
        temp_model
    )

    avg_time = measure_base_inference_time(
        temp_model,
        new_client_eval_loaders
    )

    runtime_rows.append({
        "Method": "RAS-FL personalized",
        "Activated": f"remove_{remove_p}",
        "Params": params,
        "FLOPs": flops,
        "Time(s)": avg_time,
        "Acc.": get_acc_from_avg(
            "RAS-FL",
            "ras_fl_personalized_importance",
            remove_p,
            "none"
        ),
    })


for mode_name, structured_mask in loaded_models["feddrop_roadmap"].items():
    if mode_name == "full":
        remove_p = 0
    elif mode_name.startswith("feddrop_remove_"):
        remove_p = int(mode_name.replace("feddrop_remove_", ""))
    else:
        continue

    temp_model = make_masked_base_model(
        loaded_models["feddrop_model"],
        structured_mask
    )

    params, flops = profile_effective_params_flops(
        temp_model
    )

    avg_time = measure_base_inference_time(
        temp_model,
        new_client_eval_loaders
    )

    runtime_rows.append({
        "Method": "Federated Dropout-style",
        "Activated": f"remove_{remove_p}",
        "Params": params,
        "FLOPs": flops,
        "Time(s)": avg_time,
        "Acc.": get_acc_from_avg(
            "FedDrop",
            "random_boundary",
            remove_p,
            "none"
        ),
    })


for mode_name, structured_mask in loaded_models["ofa_roadmap"].items():
    if mode_name == "full":
        remove_p = 0
    elif mode_name.startswith("ofa_remove_"):
        remove_p = int(mode_name.replace("ofa_remove_", ""))
    else:
        continue

    temp_model = make_masked_base_model(
        loaded_models["ofa_model"],
        structured_mask
    )

    params, flops = profile_effective_params_flops(
        temp_model
    )

    avg_time = measure_base_inference_time(
        temp_model,
        new_client_eval_loaders
    )

    runtime_rows.append({
        "Method": "OFA-style",
        "Activated": f"remove_{remove_p}",
        "Params": params,
        "FLOPs": flops,
        "Time(s)": avg_time,
        "Acc.": get_acc_from_avg(
            "OFA",
            "predefined_nested_width",
            remove_p,
            "none"
        ),
    })


for mode_name, structured_mask in loaded_models["slex_roadmap"].items():
    if mode_name == "full":
        remove_p = 0
    elif mode_name.startswith("slex_remove_"):
        remove_p = int(mode_name.replace("slex_remove_", ""))
    else:
        continue

    for exit_name in SLEX_EXITS:
        temp_model = make_masked_slex_model(
            loaded_models["slex_model"],
            structured_mask
        )

        params, flops = profile_effective_params_flops(
            temp_model,
            slex_exit=exit_name
        )

        avg_time = measure_slex_inference_time(
            temp_model,
            new_client_eval_loaders,
            exit_name
        )

        runtime_rows.append({
            "Method": "SLEX-style",
            "Activated": f"remove_{remove_p}_{exit_name}",
            "Params": params,
            "FLOPs": flops,
            "Time(s)": avg_time,
            "Acc.": get_acc_from_avg(
                "SLEX",
                "pure_slex_width_early_exit",
                remove_p,
                exit_name
            ),
        })


for remove_p, width_mult in SLIMMABLE_WIDTHS.items():
    width_mult = max(float(width_mult), MIN_LAYER_ACTIVE_RATIO)

    temp_model = make_slimmable_model_at_width(
        loaded_models["slimmable_model"],
        width_mult
    )

    params, flops = get_slimmable_runtime_from_checkpoint(
        loaded_models["slimmable_ckpt"],
        remove_p,
        width_mult
    )

    avg_time = measure_base_inference_time(
        temp_model,
        new_client_eval_loaders
    )

    runtime_rows.append({
        "Method": "Slimmable",
        "Activated": f"remove_{remove_p}",
        "Params": params,
        "FLOPs": flops,
        "Time(s)": avg_time,
        "Acc.": get_acc_from_avg(
            "Slimmable",
            "slimmable_width",
            remove_p,
            "none"
        ),
    })


runtime_table_all = pd.DataFrame(runtime_rows)

runtime_table_all["Params_fmt"] = runtime_table_all["Params"].apply(fmt_params)
runtime_table_all["FLOPs_fmt"] = runtime_table_all["FLOPs"].apply(fmt_flops)
runtime_table_all["Time_fmt"] = runtime_table_all["Time(s)"].apply(
    lambda x: f"{x:.4f}"
)
runtime_table_all["Acc_fmt"] = runtime_table_all["Acc."].apply(
    lambda x: f"{100.0 * x:.2f}" if pd.notna(x) else "NA"
)

paper_runtime_table_all = runtime_table_all[
    ["Method", "Activated", "Params_fmt", "FLOPs_fmt", "Time_fmt", "Acc_fmt"]
].rename(columns={
    "Params_fmt": "Params",
    "FLOPs_fmt": "FLOPs",
    "Time_fmt": "Time(s)",
    "Acc_fmt": "Acc."
})

print("\nFull runtime table over all activated parts:")
display(paper_runtime_table_all)

runtime_table_all.to_csv(
    RUNTIME_TABLE_CSV,
    index=False
)

print("\nSaved runtime table:")
print(RUNTIME_TABLE_CSV)


# Prepare the compact runtime view

compact_specs = [
    ("FedAvg", "full"),
    ("FedProx", "full"),
    ("Federated Dropout-style", f"remove_{SELECTED_REMOVE_PERCENT}"),
    ("OFA-style", f"remove_{SELECTED_REMOVE_PERCENT}"),
    ("SLEX-style", f"remove_{SELECTED_REMOVE_PERCENT}_{SELECTED_SLEX_EXIT}"),
    ("Slimmable", f"remove_{SELECTED_REMOVE_PERCENT}"),
    ("RAS-FL global", f"remove_{SELECTED_REMOVE_PERCENT}"),
    ("RAS-FL personalized", f"remove_{SELECTED_REMOVE_PERCENT}"),
]

compact_chart_df = runtime_table_all.copy()

selected_rows = []

for method, activated_key in compact_specs:
    temp = compact_chart_df[
        (compact_chart_df["Method"] == method)
        & (compact_chart_df["Activated"].str.contains(activated_key, regex=False))
    ]

    if len(temp) > 0:
        selected_rows.append(temp.iloc[0])
    else:
        print(f"Warning: compact row not found for {method} / {activated_key}")

compact_chart_df = pd.DataFrame(selected_rows).reset_index(drop=True)

compact_chart_df["Params_M"] = compact_chart_df["Params"] / 1e6
compact_chart_df["FLOPs_G"] = compact_chart_df["FLOPs"] / 1e9
compact_chart_df["Accuracy_%"] = compact_chart_df["Acc."] * 100.0

print("\nCompact chart dataframe:")
display(
    compact_chart_df[
        ["Method", "Activated", "Params", "Params_M", "FLOPs", "FLOPs_G", "Time(s)", "Accuracy_%"]
    ]
)

compact_chart_df.to_csv(
    "cifar10_vgg_runtime_compact_chart_data_with_slimmable.csv",
    index=False
)


# Runtime charts

plt.figure(figsize=(11, 5.5))

plt.bar(
    compact_chart_df["Method"],
    compact_chart_df["Params_M"],
    edgecolor="black",
    linewidth=1.0
)

plt.ylabel("Active Parameters (M)", fontweight="bold")
plt.xlabel("Method", fontweight="bold")
plt.title(
    f"Activated Parameters by Method "
    f"(remove={SELECTED_REMOVE_PERCENT}%, SLEX={SELECTED_SLEX_EXIT})",
    fontweight="bold"
)

plt.xticks(rotation=30, ha="right", fontweight="bold")
plt.grid(axis="y", alpha=0.3)

for i, v in enumerate(compact_chart_df["Params_M"]):
    plt.text(
        i,
        v,
        f"{v:.2f}",
        ha="center",
        va="bottom",
        fontsize=9
    )

plt.tight_layout()
plt.savefig(
    "cifar10_vgg_runtime_chart_params_with_slimmable.png",
    dpi=400,
    bbox_inches="tight"
)
plt.savefig(
    "cifar10_vgg_runtime_chart_params_with_slimmable.pdf",
    bbox_inches="tight"
)
plt.show()


plt.figure(figsize=(11, 5.5))

plt.bar(
    compact_chart_df["Method"],
    compact_chart_df["FLOPs_G"],
    edgecolor="black",
    linewidth=1.0
)

plt.ylabel("Active FLOPs (G)", fontweight="bold")
plt.xlabel("Method", fontweight="bold")
plt.title(
    f"Activated FLOPs by Method "
    f"(remove={SELECTED_REMOVE_PERCENT}%, SLEX={SELECTED_SLEX_EXIT})",
    fontweight="bold"
)

plt.xticks(rotation=30, ha="right", fontweight="bold")
plt.grid(axis="y", alpha=0.3)

for i, v in enumerate(compact_chart_df["FLOPs_G"]):
    plt.text(
        i,
        v,
        f"{v:.3f}",
        ha="center",
        va="bottom",
        fontsize=9
    )

plt.tight_layout()
plt.savefig(
    "cifar10_vgg_runtime_chart_flops_with_slimmable.png",
    dpi=400,
    bbox_inches="tight"
)
plt.savefig(
    "cifar10_vgg_runtime_chart_flops_with_slimmable.pdf",
    bbox_inches="tight"
)
plt.show()


plt.figure(figsize=(11, 5.5))

plt.bar(
    compact_chart_df["Method"],
    compact_chart_df["Time(s)"],
    edgecolor="black",
    linewidth=1.0
)

plt.ylabel("Average Inference Time (s)", fontweight="bold")
plt.xlabel("Method", fontweight="bold")
plt.title(
    f"Average Inference Time by Method "
    f"(remove={SELECTED_REMOVE_PERCENT}%, SLEX={SELECTED_SLEX_EXIT})",
    fontweight="bold"
)

plt.xticks(rotation=30, ha="right", fontweight="bold")
plt.grid(axis="y", alpha=0.3)

for i, v in enumerate(compact_chart_df["Time(s)"]):
    plt.text(
        i,
        v,
        f"{v:.4f}",
        ha="center",
        va="bottom",
        fontsize=9
    )

plt.tight_layout()
plt.savefig(
    "cifar10_vgg_runtime_chart_time_with_slimmable.png",
    dpi=400,
    bbox_inches="tight"
)
plt.savefig(
    "cifar10_vgg_runtime_chart_time_with_slimmable.pdf",
    bbox_inches="tight"
)
plt.show()


plt.figure(figsize=(8.5, 6.5))

plt.scatter(
    compact_chart_df["Params_M"],
    compact_chart_df["Accuracy_%"],
    s=90,
    edgecolor="black",
    linewidth=1.0
)

for _, row in compact_chart_df.iterrows():
    plt.annotate(
        row["Method"],
        (row["Params_M"], row["Accuracy_%"]),
        textcoords="offset points",
        xytext=(5, 5),
        fontsize=9
    )

plt.xlabel("Active Parameters (M)", fontweight="bold")
plt.ylabel("Accuracy (%)", fontweight="bold")
plt.title("Accuracy vs Activated Parameters", fontweight="bold")
plt.grid(alpha=0.3)
plt.tight_layout()
plt.savefig(
    "cifar10_vgg_runtime_chart_accuracy_vs_params_with_slimmable.png",
    dpi=400,
    bbox_inches="tight"
)
plt.savefig(
    "cifar10_vgg_runtime_chart_accuracy_vs_params_with_slimmable.pdf",
    bbox_inches="tight"
)
plt.show()


plt.figure(figsize=(8.5, 6.5))

plt.scatter(
    compact_chart_df["FLOPs_G"],
    compact_chart_df["Accuracy_%"],
    s=90,
    edgecolor="black",
    linewidth=1.0
)

for _, row in compact_chart_df.iterrows():
    plt.annotate(
        row["Method"],
        (row["FLOPs_G"], row["Accuracy_%"]),
        textcoords="offset points",
        xytext=(5, 5),
        fontsize=9
    )

plt.xlabel("Active FLOPs (G)", fontweight="bold")
plt.ylabel("Accuracy (%)", fontweight="bold")
plt.title("Accuracy vs Activated FLOPs", fontweight="bold")
plt.grid(alpha=0.3)
plt.tight_layout()
plt.savefig(
    "cifar10_vgg_runtime_chart_accuracy_vs_flops_with_slimmable.png",
    dpi=400,
    bbox_inches="tight"
)
plt.savefig(
    "cifar10_vgg_runtime_chart_accuracy_vs_flops_with_slimmable.pdf",
    bbox_inches="tight"
)
plt.show()


plt.figure(figsize=(8.5, 6.5))

plt.scatter(
    compact_chart_df["Time(s)"],
    compact_chart_df["Accuracy_%"],
    s=90,
    edgecolor="black",
    linewidth=1.0
)

for _, row in compact_chart_df.iterrows():
    plt.annotate(
        row["Method"],
        (row["Time(s)"], row["Accuracy_%"]),
        textcoords="offset points",
        xytext=(5, 5),
        fontsize=9
    )

plt.xlabel("Average Inference Time (s)", fontweight="bold")
plt.ylabel("Accuracy (%)", fontweight="bold")
plt.title("Accuracy vs Average Inference Time", fontweight="bold")
plt.grid(alpha=0.3)
plt.tight_layout()
plt.savefig(
    "cifar10_vgg_runtime_chart_accuracy_vs_time_with_slimmable.png",
    dpi=400,
    bbox_inches="tight"
)
plt.savefig(
    "cifar10_vgg_runtime_chart_accuracy_vs_time_with_slimmable.pdf",
    bbox_inches="tight"
)
plt.show()


print("\nSaved runtime charts:")
print("cifar10_vgg_runtime_chart_params_with_slimmable.png / .pdf")
print("cifar10_vgg_runtime_chart_flops_with_slimmable.png / .pdf")
print("cifar10_vgg_runtime_chart_time_with_slimmable.png / .pdf")
print("cifar10_vgg_runtime_chart_accuracy_vs_params_with_slimmable.png / .pdf")
print("cifar10_vgg_runtime_chart_accuracy_vs_flops_with_slimmable.png / .pdf")
print("cifar10_vgg_runtime_chart_accuracy_vs_time_with_slimmable.png / .pdf")

print("\nCell 10 completed successfully.")
